# Persian Collocation Extraction from the Hamshahri Corpus

This notebook extracts statistically strong Persian **bigram collocations** from the
[Hamshahri](https://dbrg.upv.es/) newspaper corpus and produces a ranked dataset in which every
collocation is accompanied by real example sentences drawn from the corpus.

**Goal:** identify Persian word pairs that co-occur significantly more often than chance,
rank them with a weighted combination of four association measures, and attach authentic
corpus sentences so the results can be used directly in end-user applications.

Pipeline:

```
.ham files -> parse and preserve RAW text
           -> repair mi/nemi spacing before normalization
           -> lightweight Persian character normalization
           -> Sentsplit sentence segmentation (fa)
           -> Stanza tokenization, lemmatization, POS tagging per sentence
           -> HooshvareLab NER (PER only) per sentence
           -> character-offset alignment of NER spans to Stanza words
           -> post-tagging stopword / clitic / copula filtering
           -> adjacent bigram candidate extraction
           -> association scores (PMI, t-score, logDice, LLR)
           -> frequency and document-frequency filtering
           -> majority-vote POS labels
           -> weighted final ranking
           -> real example sentences from RAW article text

The notebook is designed for deterministic 50-document quality checks first and
resumable full-corpus processing afterward.
```

## 1. Environment and Dependencies

Main libraries and their roles:

| Library | Role |
|---|---|
| **Stanza** | Neural Persian tokenization, lemmatization, and Universal Dependencies POS tagging |
| **Hazm** | Persian sentence tokenizer (raw sentence pool) |
| **Transformers** | Runs the HooshvareLab Persian BERT NER model (`HooshvareLab/bert-fa-base-uncased`) for PERSON detection |
| **NumPy / Pandas** | Counters, data frames, scoring, and analysis |
| **PyArrow** | Fast columnar checkpoints (parquet) |
| **OpenPyXL** | Excel export of the final dataset |

In [35]:
# Run once per Kaggle session.
!pip install -q stanza hazm transformers sentencesplit openpyxl pyarrow tqdm


In [ ]:
import re
import os
import json
import time
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pickle

CONFIG = {
    "SEED": 42,
    "N_DOCS_SUBSET": None,
    "SAMPLE_SIZE": 50,
    "WINDOW_SIZE": 2,
    "MIN_FREQUENCY": 10,
    "MIN_DOC_FREQUENCY": 3,
    "FINAL_MIN_FREQUENCY": 15,
    "FINAL_MIN_DOC_FREQUENCY": 5,
    "TOP_N_FINAL": 5000,
    "N_EXAMPLES_PER_PAIR": 5,
    "EXAMPLE_MIN_WORDS": 7,
    "EXAMPLE_MAX_WORDS": 25,
    "NER_MIN_SCORE": 0.50,
    "KAGGLE_INPUT_DIR": "/kaggle/input",
    "WORKING_DIR": "/kaggle/working",
    "CHECKPOINT_DIR": "/kaggle/working/checkpoints_stanza",
}

random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])
Path(CONFIG["WORKING_DIR"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["CHECKPOINT_DIR"]).mkdir(parents=True, exist_ok=True)
print(CONFIG)


{'SEED': 42, 'N_DOCS_SUBSET': None, 'SAMPLE_SIZE': 50, 'WINDOW_SIZE': 2, 'MIN_FREQUENCY': 10, 'MIN_DOC_FREQUENCY': 3, 'FINAL_MIN_FREQUENCY': 15, 'FINAL_MIN_DOC_FREQUENCY': 5, 'TOP_N_FINAL': 5000, 'N_EXAMPLES_PER_PAIR': 5, 'EXAMPLE_MIN_WORDS': 7, 'EXAMPLE_MAX_WORDS': 25, 'NER_MIN_SCORE': 0.5, 'KAGGLE_INPUT_DIR': '/kaggle/input', 'WORKING_DIR': '/kaggle/working', 'CHECKPOINT_DIR': '/kaggle/working/checkpoints_stanza'}


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [4]:
# Find every .ham file, wherever Kaggle mounted the dataset.
ham_files = sorted(Path(CONFIG["KAGGLE_INPUT_DIR"]).rglob("*.ham"))
assert ham_files, "No .ham files found. Add the Hamshahri dataset via 'Add Input' first."
print("Found", len(ham_files), ".ham files")
print("Example path:", ham_files[0])


Found 5375 .ham files
Example path: /kaggle/input/datasets/ehsankhani/hamshahri-corpus/HamshahriData/HamshahriCorpus/2003/HAM2-811011-007.ham


In [5]:
FILENAME_DATE_RE = re.compile(r"HAM\d*-(\d{6})-\d+")
CKPT_CORPUS = Path(CONFIG["CHECKPOINT_DIR"]) / "corpus_raw.parquet"

def read_file_safely(path: Path) -> str:
    raw_bytes = path.read_bytes()
    for enc in ["utf-8", "windows-1256", "utf-16"]:
        try:
            return raw_bytes.decode(enc).strip()
        except UnicodeDecodeError:
            continue
    return raw_bytes.decode("utf-8", errors="replace").strip()

def parse_ham_file(path: Path) -> dict:
    text = read_file_safely(path)
    lines = [l for l in text.split("\n") if l.strip()]
    title = lines[0] if lines else None
    body = "\n".join(lines[1:]) if len(lines) > 1 else ""
    m = FILENAME_DATE_RE.search(path.name)
    return {
        "doc_id": path.stem,
        "title": title,
        "text": text,
        "body": body,
        "date_code": m.group(1) if m else None,
        "year_folder": path.parent.name,
    }

if CKPT_CORPUS.exists():
    corpus_df = pd.read_parquet(CKPT_CORPUS)
    print(f"Loaded corpus checkpoint: {len(corpus_df)} documents")
else:
    parsed_docs = []
    for p in ham_files:
        doc = parse_ham_file(p)
        if len(doc["text"]) > 0:
            parsed_docs.append(doc)
            
    corpus_df = pd.DataFrame(parsed_docs).reset_index(drop=True)
    corpus_df.to_parquet(CKPT_CORPUS, index=False)
    print(f"Parsed and checkpointed: {len(corpus_df)} documents")

print("Missing dates:", corpus_df["date_code"].isna().sum())
corpus_df.head(2)

Parsed and checkpointed: 5375 documents
Missing dates: 0


,doc_id,title,text,body,date_code,year_folder
0,HAM2-811011-007,جودي ابوت، نابغه كاغذي,جودي ابوت، نابغه كاغذي\nسميه نصيري ها\nجودي اب...,سميه نصيري ها\nجودي ابوت را سال ها است كه مي ش...,811011,2003
1,HAM2-811011-023,برنامه ريزي براي مرگ,برنامه ريزي براي مرگ\nنگاهي به استرس و ساز و ك...,نگاهي به استرس و ساز و كارهاي آن در انسان ها\n...,811011,2003


In [6]:
print("Documents:", len(corpus_df))
print("Empty text:", corpus_df["text"].str.strip().eq("").sum())
print("Missing date:", corpus_df["date_code"].isna().sum())
print("Unique doc_id:", corpus_df["doc_id"].nunique())
print("Years:", corpus_df["date_code"].str[:2].value_counts().sort_index())

Documents: 5375
Empty text: 0
Missing date: 0
Unique doc_id: 5375
Years: date_code
81     199
82    1225
83    1215
84    1379
85    1261
86      96
Name: count, dtype: int64


In [7]:
print(corpus_df.iloc[9]["text"][:500])

پرسپوليس ديروز در نقش جهان بازي را به 
اصفهاني ها باخت
گزارش خبرنگار همشهري از اصفهان و حاشيه هاي بازي ديروز را در
همين صفحه و صفحه 13 بخوانيد
عكس: محمدرضا شاهرخي نژاد



حاشيه بازي _ پرسپوليس سپاهان 
داود فنايي شيشه رختكن ورزشگاه را نيز با عصبانيت شكست 
پس از اخراج ناصر ابراهيمي او به سمت كمك داور رفت و فحاشي كرد
آرش راهبر، خبرنگار اعزامي همشهري به اصفهان: ورزشگاه نقش جهان تا ساعات اوليه صبح درگير
آماده سازي و به دست آوردن شرايط بهتر بود. گروه سازندگان و پيمانكاران در مدت يك هفته 
بسياري از امك


## 2. Corpus Preparation

### Normalizing verb-prefix spacing before normalization

Hamshahri text often contains a regular space between verbal prefixes and verbs. This is repaired **before**
character normalization so tokenization sees the intended joined form (`می رفت` → `می‌رفت`).

Corpus loading preserves the **RAW article text** alongside the normalized version: raw text is kept
untouched because the final example sentences shown to end users must come verbatim from the corpus,
while normalized text is used for all tagging and extraction steps.

In [8]:
# Persian text normalization: spacing + Unicode normalization

ZWNJ = "\u200c"

# Arabic/Persian character normalization
CHAR_TRANSLATION = str.maketrans({
    "ي": "ی",
    "ى": "ی",
    "ك": "ک",
    "ة": "ه",
    "ۀ": "ه",
})

VERB_PREFIX_RE = re.compile(
    r"(?<!\S)(ن?می)[ \t]+(?=\S)",
    re.UNICODE
)

def normalize_persian_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Normalize Arabic/Persian character variants
    text = text.translate(CHAR_TRANSLATION)

    # Normalize different whitespace characters
    text = re.sub(r"[\u00A0\u2000-\u200A\u202F]", " ", text)

    # Normalize می / نمی + verb → می‌ / نمی‌ + verb
    text = VERB_PREFIX_RE.sub(lambda m: m.group(1) + ZWNJ, text)

    # Normalize repeated spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


corpus_df["text_norm"] = corpus_df["text"].map(normalize_persian_text)
corpus_df["body_norm"] = corpus_df["body"].map(normalize_persian_text)

print("Documents:", len(corpus_df))
print("Empty normalized texts:", corpus_df["text_norm"].eq("").sum())

Documents: 5375
Empty normalized texts: 0


In [9]:
ZWNJ = "\u200c"

# Normalize Arabic/Persian character variants
CHAR_TRANSLATION = str.maketrans({
    "ي": "ی",
    "ى": "ی",
    "ك": "ک",
    "ۀ": "ه",
})

# فقط می / نمی
VERB_PREFIX_RE = re.compile(
    r"(?<![\w\u200c])((?:ن)?می|(?:ن)?مي)\s+(?=\S)",
    re.UNICODE
)

def normalize_persian_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Arabic → Persian characters
    text = text.translate(CHAR_TRANSLATION)

    # Normalize whitespace
    text = re.sub(r"[\u00A0\u2000-\u200A\u202F]", " ", text)

    # می + فعل / نمی + فعل
    text = VERB_PREFIX_RE.sub(
        lambda m: ("نمی" if m.group(1).startswith("ن") else "می") + ZWNJ,
        text
    )

    # Multiple spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


corpus_df["text_norm"] = corpus_df["text"].map(normalize_persian_text)
corpus_df["body_norm"] = corpus_df["body"].map(normalize_persian_text)

In [10]:
# Show ONLY actual می / نمی spacing corrections

CHECK_RE = re.compile(
    r"(?<![\w\u200c])(ن?(?:می|مي))\s+([^\s]+)",
    re.UNICODE
)

MAX_SAMPLES = 30
shown = 0

print("=" * 85)
print(f"{'#':<3} | {'BEFORE':<30} | {'AFTER':<30}")
print("=" * 85)

for raw in corpus_df["text"]:

    for match in CHECK_RE.finditer(raw):

        before = match.group(0)
        after = normalize_persian_text(before)

        # Only real normalization changes
        if before != after:
            shown += 1
            print(f"{shown:<3} | {after:<30} | {before:<30}")

            if shown >= MAX_SAMPLES:
                break

    if shown >= MAX_SAMPLES:
        break

print("=" * 85)
print(f"Displayed: {shown} examples")

#   | BEFORE                         | AFTER                         
1   | می‌شناسم،                      | مي شناسم،                     
2   | می‌خواست                       | مي خواست                      
3   | می‌خواندم                      | مي خواندم                     
4   | می‌کند                         | مي كند                        
5   | می‌اندازد.                     | مي اندازد.                    
6   | می‌خواست                       | مي خواست                      
7   | می‌آمد                         | مي آمد                        
8   | می‌کرد.                        | مي كرد.                       
9   | می‌تواند                       | مي تواند                      
10  | می‌توانست                      | مي توانست                     
11  | می‌شود                         | مي شود                        
12  | می‌خرد                         | مي خرد                        
13  | می‌فروشد.                      | مي فروشد.                     
14  | می‌شود        

### Character normalization

Apply lightweight character normalization while preserving punctuation, stopwords, and sentence structure
for downstream tagging:

- Unification of Arabic/Persian character variants (`ي`→`ی`, `ك`→`ک`, Arabic digits, etc.)
- Unicode normalization and whitespace cleanup
- Correction of `می` / `نمی` prefix spacing (joined with ZWNJ)
- Punctuation and sentence boundaries are preserved so segmentation remains reliable

In [11]:
CKPT_NORM = Path(CONFIG["CHECKPOINT_DIR"]) / "corpus_normalized.parquet"

PERSIAN_CHAR_MAP = str.maketrans({
    "ي": "ی",
    "ى": "ی",
    "ك": "ک",
    "ۀ": "هٔ",
    "ة": "ه",
    "ؤ": "ؤ",
    "إ": "ا",
    "أ": "ا",
    "ٱ": "ا",
})

DIACRITICS_RE = re.compile(r"[\u064B-\u065F\u0670]")
MULTISPACE_RE = re.compile(r"[ \t\r]+")

# فقط می / نمی + فاصله → ZWNJ
VERB_PREFIX_RE = re.compile(
    r"(?<![\w\u200c])(ن?(?:می|مي))\s+(?=\S)",
    re.UNICODE
)

def normalize_persian_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Arabic → Persian
    text = text.translate(PERSIAN_CHAR_MAP)

    # Remove diacritics
    text = DIACRITICS_RE.sub("", text)

    # Remove zero-width joiner
    text = text.replace("\u200d", "")

    # Normalize NBSP
    text = text.replace("\u00a0", " ")

    # می / نمی + فعل
    text = VERB_PREFIX_RE.sub(
        lambda m: ("نمی" if m.group(1).startswith("ن") else "می") + "\u200c",
        text
    )

    # Normalize spaces
    text = MULTISPACE_RE.sub(" ", text)

    return text.strip()


if CKPT_NORM.exists():
    corpus_df = pd.read_parquet(CKPT_NORM)
    print(f"Loaded normalized checkpoint: {len(corpus_df):,} documents")

else:
    corpus_df["norm_text"] = corpus_df["text"].map(normalize_persian_text)
    corpus_df["norm_body"] = corpus_df["body"].map(normalize_persian_text)

    corpus_df.to_parquet(CKPT_NORM, index=False)

    print(f"Normalized and checkpointed: {len(corpus_df):,} documents")

Normalized and checkpointed: 5,375 documents


In [12]:
# Final preprocessing inspection

DOC_INDEX = 0
N_LINES = 8

raw_text = corpus_df.loc[DOC_INDEX, "text"]
norm_text = corpus_df.loc[DOC_INDEX, "norm_text"]

raw_lines = [line.strip() for line in raw_text.splitlines() if line.strip()]
norm_lines = [line.strip() for line in norm_text.splitlines() if line.strip()]

print("=" * 90)
print(f"Document: {corpus_df.loc[DOC_INDEX, 'doc_id']}")
print("=" * 90)

print("\nBEFORE NORMALIZATION")
print("-" * 90)

for line in raw_lines[:N_LINES]:
    print(line)

print("\nAFTER NORMALIZATION")
print("-" * 90)

for line in norm_lines[:N_LINES]:
    print(line)

print("\n" + "=" * 90)
print("Characters:", len(raw_text), "→", len(norm_text))
print("Changed:", raw_text != norm_text)
print("=" * 90)

Document: HAM2-811011-007

BEFORE NORMALIZATION
------------------------------------------------------------------------------------------
جودي ابوت، نابغه كاغذي
سميه نصيري ها
جودي ابوت را سال ها است كه مي شناسم، دقيقا از دوران نوجواني. شايد در دوران پرفراز
و نشيب نوجواني دلم مي خواست جاي او باشم. خيلي از لحظاتي كه ماجراهايش را مي خواندم
دوست داشتم دردسرهاي او مال من باشد. جودي دختري است كه اصولا دردسر درست مي كند و
اتفاقا دردسرهايش اطرافيان را هم به دردسر مي اندازد. خوب است قدري باانصاف باشيم.
شايد از اين جهت دلم مي خواست جاي جودي باشم كه خداوند در تمام لحظات زندگي يار و ياورش
بود. در تمام لحظات سياه و بغرنج زندگي خداوند به ياري اش مي آمد و اوضاع اش سروسامان

AFTER NORMALIZATION
------------------------------------------------------------------------------------------
جودی ابوت، نابغه کاغذی
سمیه نصیری ها
جودی ابوت را سال ها است که می‌شناسم، دقیقا از دوران نوجوانی. شاید در دوران پرفراز
و نشیب نوجوانی دلم می‌خواست جای او باشم. خیلی از لحظاتی که ماجراهایش را می‌خواندم
دوست داشتم دردسرهای

In [13]:
print("Documents:", len(corpus_df))
print("Empty normalized:", corpus_df["norm_text"].eq("").sum())
print("Changed documents:", (corpus_df["text"] != corpus_df["norm_text"]).sum())
print("Missing normalized:", corpus_df["norm_text"].isna().sum())

Documents: 5375
Empty normalized: 0
Changed documents: 5375
Missing normalized: 0


## 3. Sentence Segmentation

Collocation extraction operates at the **sentence level**: candidate pairs never cross sentence or
punctuation boundaries, because words co-occurring in different sentences are not collocations.
Accurate Persian sentence segmentation (Sentsplit, `fa`) therefore directly controls the quality of
both candidate extraction and the example sentences.

### Raw sentence pool

These examples are extracted from the original article body, using the same Persian sentence segmenter as
the tagging pipeline. The first body sentence is skipped to avoid title/byline artifacts. Only sentences
within the configured length range are retained.

In [14]:
try:
    from hazm import SentenceTokenizer
    _hazm_sent_tokenizer = SentenceTokenizer()
except Exception:
    _hazm_sent_tokenizer = None

CKPT_RAW_SENTS = Path(CONFIG["CHECKPOINT_DIR"]) / "raw_sentence_pool.jsonl"

SENT_END_RE = re.compile(r"([.!?؟]+)(\s+|$)")

def regex_split_sentences(text):
    if not isinstance(text, str) or not text.strip():
        return []
    parts, last = [], 0
    for m in SENT_END_RE.finditer(text):
        end = m.end(1)
        if chunk := text[last:end].strip():
            parts.append(chunk)
        last = m.end()
    if tail := text[last:].strip():
        parts.append(tail)
    return parts

def split_persian_sentences(text):
    if not isinstance(text, str) or not text.strip():
        return []
    if _hazm_sent_tokenizer:
        try:
            sents = [s.strip() for s in _hazm_sent_tokenizer.tokenize(text) if s.strip()]
            if sents:
                return sents
        except Exception:
            pass
    return regex_split_sentences(text)

def good_example_sentences(body, min_words, max_words):
    sentences = split_persian_sentences(body)
    return [
        s for s in sentences
        if min_words <= len(s.split()) <= max_words
    ]

def build_raw_sentence_pool(df, ckpt_path):
    done_ids = set()

    if ckpt_path.exists():
        with open(ckpt_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    done_ids.add(json.loads(line)["doc_id"])
                except Exception:
                    pass

    remaining = df[~df["doc_id"].isin(done_ids)]

    with open(ckpt_path, "a", encoding="utf-8") as f:
        for _, row in remaining.iterrows():
            sentences = good_example_sentences(
                row["norm_body"],
                CONFIG["EXAMPLE_MIN_WORDS"],
                CONFIG["EXAMPLE_MAX_WORDS"]
            )
            f.write(json.dumps({
                "doc_id": row["doc_id"],
                "sentences": sentences
            }, ensure_ascii=False) + "\n")

    print(f"Processed: {len(remaining):,} documents")

def load_raw_sentence_pool(ckpt_path):
    pool = {}
    with open(ckpt_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            pool[record["doc_id"]] = record["sentences"]
    return pool

build_raw_sentence_pool(corpus_df, CKPT_RAW_SENTS)

raw_sentence_pool = load_raw_sentence_pool(CKPT_RAW_SENTS)
n_total_examples = sum(map(len, raw_sentence_pool.values()))

print(f"Documents: {len(raw_sentence_pool):,}")
print(f"Candidate sentences: {n_total_examples:,}")

Processed: 5,375 documents
Documents: 5,375
Candidate sentences: 41,663


In [15]:
rng = random.Random(CONFIG["SEED"])
MAX_SAMPLES = 25

examples = [
    (doc_id, sent, len(sent.split()))
    for doc_id, sents in raw_sentence_pool.items()
    for sent in sents
]

rng.shuffle(examples)

print(f"{'#':<3} | {'DOC ID':<18} | {'WORDS':<6} | SENTENCE")
print("=" * 120)

for i, (doc_id, sent, word_count) in enumerate(examples[:MAX_SAMPLES], 1):
    print(f"{i:<3} | {doc_id:<18} | {word_count:<6} | {sent}")

print("=" * 120)
print(f"Displayed: {min(MAX_SAMPLES, len(examples))}")
print(f"Total candidate sentences: {len(examples):,}")

#   | DOC ID             | WORDS  | SENTENCE
1   | HAM2-811219-040    | 21     | براین اساس، عملیات اجرایی این طرح شروع و پیش بینی می‌شود تا نیمه دوم سال 2005 میلادی به بهره برداری برسد.
2   | HAM2-820330-018    | 19     | جارو زدن، ظرف شستن، البته از حق نگذریم اینجا هم مدرک لیسانس افاقه کرد و از بیگاری معاف شدیم.
3   | HAM2-820526-002    | 21     | به باور کارشناسان رشد ناگهانی قیمت سهام خودرو ظرف 3 ماه گذشته از مهم ترین دلایل انفجار بورس تهران بوده است.
4   | HAM2-821129-103    | 22     | گروه شهری: شورای اسلامی شهر تهران، پرداخت یک میلیون تومان وام قرض الحسنه ازدواج را برای زوج های جوان تهرانی تصویب کرد.
5   | HAM2-850421-069    | 16     | همه آماده هستند تا سوال های کنکور پخش شود تا شروع به تست زدن بکنند، اما...
6   | HAM2-850316-089    | 13     | برای پاسخ به این پرسش باید به تاریخچه آموزش در صنعت توریسم بازگردیم.
7   | HAM2-850210-073    | 11     | * صدام حسین دیروز در زندان 69سالگی خود را جشن گرفت.
8   | HAM2-850930-117    | 19     | می‌خواهم بگویم ناشر آثار حکمی باید حکیم باشد 

## 4. Tokenization, Lemmatization and POS Tagging

Sentence boundaries are provided by the dedicated Persian sentence segmenter.
[Stanza](https://stanfordnlp.github.io/stanza/) is used for Persian tokenization, lemmatization, and
Universal Dependencies (UPOS) tagging.

- **Tokenization:** Stanza's neural Persian tokenizer handles ZWNJ-joined forms and multi-word tokens.
- **Lemmatization:** inflected surface forms map to a canonical lemma, which is essential for Persian
  where verbs and nouns are highly inflected — counting surface forms would fragment frequencies.
- **POS tagging:** Universal Dependencies tags (`NOUN`, `ADJ`, `VERB`, ...) define which adjacent pairs
  are plausible collocations.

In [16]:
import stanza
import torch

try:
    stanza.Pipeline(
        lang="fa",
        processors="tokenize,mwt,pos,lemma",
        verbose=False
    )
except Exception:
    stanza.download("fa", processors="tokenize,mwt,pos,lemma", verbose=False)

nlp = stanza.Pipeline(
    lang="fa",
    processors="tokenize,mwt,pos,lemma",
    tokenize_pretokenized=False,
    use_gpu=torch.cuda.is_available(),
    verbose=False,
)

sample_text = "روزنامه‌نگاران اخبار مهمی را منتشر می‌کنند."

for sent_text in split_persian_sentences(sample_text):
    doc = nlp(sent_text)

    print(f"{'TOKEN':<20} | {'LEMMA':<20} | {'POS':<10}")
    print("=" * 55)

    for sentence in doc.sentences:
        for word in sentence.words:
            print(
                f"{word.text:<20} | "
                f"{word.lemma or word.text:<20} | "
                f"{word.upos:<10}"
            )

    print("=" * 55)

models/tokenize/perdt.pt:   0%|          | 0.00/638k [00:00<?, ?B/s]

models/mwt/perdt.pt:   0%|          | 0.00/619k [00:00<?, ?B/s]

models/pos/perdt_charlm.pt:   0%|          | 0.00/35.9M [00:00<?, ?B/s]

models/lemma/perdt_nocharlm.pt:   0%|          | 0.00/2.41M [00:00<?, ?B/s]

models/forward_charlm/conll17.pt:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

models/pretrain/conll17.pt:   0%|          | 0.00/108M [00:00<?, ?B/s]

models/backward_charlm/conll17.pt:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

TOKEN                | LEMMA                | POS       
روزنامه‌نگاران       | روزنامه‌نگار         | NOUN      
اخبار                | خبر                  | NOUN      
مهمی                 | مهم                  | ADJ       
را                   | را                   | ADP       
منتشر                | منتشر                | ADJ       
می‌کنند              | کرد                  | VERB      
.                    | .                    | PUNCT     


## 5. Named Entity Recognition

NER runs as a **separate pass** with the [HooshvareLab](https://huggingface.co/HooshvareLab/bert-fa-base-uncased)
Persian BERT token-classification model (transformers pipeline), applied independently to each sentence.
Only **PERSON** entities are used.

Because the transformer tokenizer is subword-based while extraction works on Stanza words, entity spans
are aligned back to Stanza tokens via **character offsets**. PERSON tokens are then flagged and later
**excluded from collocation candidates**: names like «محمد خاتمی» are not lexical collocations of the
language, and their high frequency in news text would otherwise dominate the rankings.

In [17]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    pipeline as hf_pipeline,
)

NER_MODEL_NAME = "HooshvareLab/bert-fa-base-uncased-ner-peyma"
NER_THRESHOLD = CONFIG.get("NER_MIN_SCORE", 0.70)
NER_MAX_TOKENS = 400
NER_OVERLAP = 80

ner_tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)
ner_model = AutoModelForTokenClassification.from_pretrained(NER_MODEL_NAME)

ner_pipe = hf_pipeline(
    "ner",
    model=ner_model,
    tokenizer=ner_tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1,
)

def normalize_ner_label(label):
    label = str(label or "").upper().strip()
    label = label.replace("-", "_")
    return label.split("_")[-1]

def detect_person_spans(sentence):
    if not isinstance(sentence, str) or not sentence.strip():
        return []

    encoded = ner_tokenizer(
        sentence,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    offsets = encoded["offset_mapping"]
    n_tokens = len(offsets)

    if n_tokens == 0:
        return []

    spans = []
    step = NER_MAX_TOKENS - NER_OVERLAP
    start_token = 0

    while start_token < n_tokens:
        end_token = min(start_token + NER_MAX_TOKENS, n_tokens)

        start_char = offsets[start_token][0]
        end_char = offsets[end_token - 1][1]

        if end_char > start_char:
            chunk = sentence[start_char:end_char]

            entities = ner_pipe(chunk)

            for ent in entities:
                label = normalize_ner_label(
                    ent.get("entity_group") or ent.get("entity")
                )
                score = float(ent.get("score", 0.0))

                if label != "PER" or score < NER_THRESHOLD:
                    continue

                start = start_char + int(ent["start"])
                end = start_char + int(ent["end"])

                spans.append({
                    "start": start,
                    "end": end,
                    "score": score,
                    "text": sentence[start:end],
                })

        if end_token >= n_tokens:
            break

        start_token += step

    # Deduplicate overlapping spans
    spans.sort(key=lambda x: (x["start"], -x["score"], -x["end"]))

    result = []

    for span in spans:
        overlap = False

        for existing in result:
            if (
                span["start"] < existing["end"]
                and span["end"] > existing["start"]
            ):
                overlap = True

                if span["score"] > existing["score"]:
                    existing.update(span)

                break

        if not overlap:
            result.append(span)

    return sorted(result, key=lambda x: x["start"])


def align_person_spans_to_tokens(doc, person_spans):
    flags = []

    for sentence in doc.sentences:
        for word in sentence.words:

            if word.start_char is None or word.end_char is None:
                flags.append(False)
                continue

            start = int(word.start_char)
            end = int(word.end_char)

            is_person = any(
                start < span["end"] and end > span["start"]
                for span in person_spans
            )

            flags.append(is_person)

    return flags


def tag_sentence(sentence):
    doc = nlp(sentence)
    person_spans = detect_person_spans(sentence)
    person_flags = align_person_spans_to_tokens(
        doc,
        person_spans
    )

    return doc, person_flags, person_spans

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/651M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/651M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased-ner-peyma
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
sample_text = (
    "احمدی نژاد در دانشگاه تهران با دوست خوب خود دیدار کرد. "
    "جودی ابوت گفت که به تهران رفت."
)

print("=" * 70)
print("SENTENCE SEGMENTATION")
print("=" * 70)

sentences = split_persian_sentences(sample_text)

for i, sent in enumerate(sentences, 1):
    print(f"{i}: {sent}")

print("\n" + "=" * 70)
print("NER + STANZA DIAGNOSTIC")
print("=" * 70)

for sent_id, sent in enumerate(sentences, 1):

    print(f"\nSENTENCE {sent_id}: {sent}")

    # Raw NER output
    raw_ner = ner_pipe(sent)

    print("\nRaw NER:")
    if raw_ner:
        for ent in raw_ner:
            print(
                f"  {ent.get('word')!r} | "
                f"label={ent.get('entity_group') or ent.get('entity')} | "
                f"score={float(ent.get('score', 0)):.3f} | "
                f"span=({ent.get('start')}, {ent.get('end')})"
            )
    else:
        print("  NO ENTITIES")

    # Our PER detector
    person_spans = detect_person_spans(sent)

    print("\nDetected PERSON spans:")
    if person_spans:
        for span in person_spans:
            print(
                f"  {span['text']!r} | "
                f"score={span['score']:.3f} | "
                f"span=({span['start']}, {span['end']})"
            )
    else:
        print("  NO PERSON")

    # Stanza
    doc = nlp(sent)

    print("\nToken / Lemma / POS / PERSON:")
    print("-" * 70)

    for sentence in doc.sentences:
        for word in sentence.words:

            start = word.start_char
            end = word.end_char

            is_person = False

            if start is not None and end is not None:
                is_person = any(
                    start < span["end"] and end > span["start"]
                    for span in person_spans
                )

            print(
                f"{word.text:<18} | "
                f"{(word.lemma or word.text):<18} | "
                f"{word.upos:<8} | "
                f"{'PER' if is_person else '-'}"
            )

    print("=" * 70)

SENTENCE SEGMENTATION
1: احمدی نژاد در دانشگاه تهران با دوست خوب خود دیدار کرد.
2: جودی ابوت گفت که به تهران رفت.

NER + STANZA DIAGNOSTIC

SENTENCE 1: احمدی نژاد در دانشگاه تهران با دوست خوب خود دیدار کرد.

Raw NER:
  'احمدی' | label=B_PER | score=0.999 | span=(0, 5)
  'نژاد' | label=I_PER | score=0.999 | span=(6, 10)
  'دانشگاه' | label=B_LOC | score=0.982 | span=(14, 21)
  'تهران' | label=I_LOC | score=0.991 | span=(22, 27)

Detected PERSON spans:
  'احمدی' | score=0.999 | span=(0, 5)
  'نژاد' | score=0.999 | span=(6, 10)

Token / Lemma / POS / PERSON:
----------------------------------------------------------------------
احمدی              | احمدی              | PROPN    | PER
نژاد               | نژاد               | PROPN    | PER
در                 | در                 | ADP      | -
دانشگاه            | دانشگاه            | PROPN    | -
تهران              | تهران              | PROPN    | -
با                 | با                 | ADP      | -
دوست               | دوست        

### Verb lemma normalization

Stanza returns lemma and Universal Dependencies POS tags. A conservative fallback is kept for
verb forms where the returned lemma is missing or encoded as a past#present pair.

Different inflected forms of the same verb («رفت», «می‌رود», «برو») should map to one canonical lemma;
otherwise the frequency of a collocation such as *اعلام + کرد* would be split across variants and
under-scored. The normalization below performs this canonical mapping explicitly.

In [20]:
# Canonical verb normalization for Persian collocation extraction

VERB_EXCEPTIONS = {
    "هست": "بودن",
    "بایست": "بایست",
}

AUX_EXCEPTIONS = {
    "است": "بودن",
    "بود": "بودن",
    "هست": "بودن",
    "خواست": "خواستن",
}

# Clear Stanza mislemmatizations where the surface form is informative.
VERB_SURFACE_FIXES = {
    "برداشت": "برداشتن",
    "برداشته": "برداشتن",
    "بردارد": "برداشتن",
    "بردارند": "برداشتن",
    "بردارید": "برداشتن",
}

def normalize_verb_lemma(lemma: str, pos: str = None, surface: str = None) -> str:
    lemma = (lemma or "").strip()
    surface = (surface or "").strip()

    if not lemma:
        return surface

    if pos == "AUX":
        return AUX_EXCEPTIONS.get(lemma, lemma)

    if pos != "VERB":
        return lemma

    # Fix clear surface-specific Stanza errors first.
    if surface in VERB_SURFACE_FIXES:
        return VERB_SURFACE_FIXES[surface]

    # Explicit lemma exceptions.
    if lemma in VERB_EXCEPTIONS:
        return VERB_EXCEPTIONS[lemma]

    # Already canonical infinitive.
    if lemma.endswith("ن"):
        return lemma

    # General Persian verb rule.
    return lemma + "ن"

In [21]:
test_text = (
    "احمدی نژاد در دانشگاه تهران اعلام کرد که کارشناسان اخبار مهمی را بررسی می‌کنند. "
    "جودی ابوت گفت اگر مشکلات به سرعت حل نشود، شرایط به سختی تغییر می‌یابد و مسئولان "
    "تصمیم جدیدی نخواهند گرفت. او نیز این تصمیمات محمد علی را به دقت پیگیری می‌کرد."
)

for sent_id, sentence in enumerate(split_persian_sentences(test_text), 1):

    doc, flags, person_spans = tag_sentence(sentence)

    print(f"\nSENTENCE {sent_id}: {sentence}")
    print("PERSON:", [x["text"] for x in person_spans])

    print(f"{'TOKEN':<16} | {'POS':<8} | {'LEMMA':<16} | {'PERSON'}")
    print("-" * 70)

    flag_index = 0

    for stanza_sentence in doc.sentences:
        for word in stanza_sentence.words:

            is_person = flags[flag_index]
            flag_index += 1

            lemma = normalize_verb_lemma(
                word.lemma or word.text,
                pos=word.upos,
                surface=word.text
            )

            print(
                f"{word.text:<16} | "
                f"{word.upos:<8} | "
                f"{lemma:<16} | "
                f"{'YES' if is_person else 'NO'}"
            )

    print("-" * 70)


SENTENCE 1: احمدی نژاد در دانشگاه تهران اعلام کرد که کارشناسان اخبار مهمی را بررسی می‌کنند.
PERSON: ['احمدی', 'نژاد']
TOKEN            | POS      | LEMMA            | PERSON
----------------------------------------------------------------------
احمدی            | PROPN    | احمدی            | YES
نژاد             | PROPN    | نژاد             | YES
در               | ADP      | در               | NO
دانشگاه          | PROPN    | دانشگاه          | NO
تهران            | PROPN    | تهران            | NO
اعلام            | NOUN     | اعلام            | NO
کرد              | VERB     | کردن             | NO
که               | SCONJ    | که               | NO
کارشناسان        | NOUN     | کارشناس          | NO
اخبار            | NOUN     | خبر              | NO
مهمی             | ADJ      | مهم              | NO
را               | ADP      | را               | NO
بررسی            | NOUN     | بررسی            | NO
می‌کنند          | VERB     | کردن             | NO
.                | PUNCT

## 6. Quality Evaluation — 50-Document Sample

Before committing to a full-corpus run (hours of GPU time), the entire pipeline is executed on a
**deterministic 50-document sample** (fixed seed). This validates segmentation, tokenization,
lemmatization, POS tagging, PERSON alignment, and candidate bigrams on data small enough to inspect
manually. Any systematic error caught here would otherwise be amplified across hundreds of thousands
of documents.

In [22]:
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

CKPT_INSPECTION = Path(CONFIG["CHECKPOINT_DIR"]) / "inspection_df.csv"
sample_docs = corpus_df.head(CONFIG["SAMPLE_SIZE"]).reset_index(drop=True)

inspection_rows = []

for _, row in tqdm(sample_docs.iterrows(), total=len(sample_docs), desc="Inspecting"):
    doc_id = row["doc_id"]
    text = row["norm_body"]

    if not isinstance(text, str) or not text.strip():
        continue

    for sent_idx, sentence in enumerate(split_persian_sentences(text)):
        doc, flags, _ = tag_sentence(sentence)
        flag_index = 0

        for stanza_sentence in doc.sentences:
            for word in stanza_sentence.words:
                is_person = flags[flag_index]
                flag_index += 1

                token = word.text or ""
                raw_lemma = word.lemma or token
                pos = word.upos or "X"

                final_lemma = normalize_verb_lemma(
                    raw_lemma,
                    pos=pos,
                    surface=token
                )

                inspection_rows.append({
                    "doc_id": doc_id,
                    "sent_idx": sent_idx,
                    "token": token,
                    "raw_lemma": raw_lemma,
                    "final_lemma": final_lemma,
                    "pos": pos,
                    "is_person": bool(is_person),
                    "is_punct": pos in {"PUNCT", "SYM"},
                })

inspection_df = pd.DataFrame(inspection_rows)
inspection_df.to_csv(
    CKPT_INSPECTION,
    index=False,
    encoding="utf-8-sig"
)

print(f"Documents : {sample_docs['doc_id'].nunique()}")
print(f"Tokens    : {len(inspection_df):,}")
print(f"PERSON    : {inspection_df['is_person'].sum():,}")
print(f"VERB      : {(inspection_df['pos'] == 'VERB').sum():,}")
print(f"AUX       : {(inspection_df['pos'] == 'AUX').sum():,}")
print(f"Saved     : {CKPT_INSPECTION}")

Inspecting:   0%|          | 0/50 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Documents : 50
Tokens    : 37,100
PERSON    : 685
VERB      : 2,811
AUX       : 1,055
Saved     : /kaggle/working/checkpoints_stanza/inspection_df.csv


In [23]:
sample_doc_ids = inspection_df["doc_id"].drop_duplicates().head(3)

for doc_id in sample_doc_ids:
    print("=" * 100)
    print(f"DOCUMENT: {doc_id}")

    full_text = corpus_df.loc[
        corpus_df["doc_id"] == doc_id, "norm_body"
    ].iloc[0]

    print("\nTEXT SAMPLE")
    print("-" * 100)
    print(full_text[:1500])

    rows = inspection_df[inspection_df["doc_id"] == doc_id]

    print("\nTOKEN ANALYSIS")
    print("-" * 100)
    print(f"{'TOKEN':<18} | {'RAW LEMMA':<18} | {'FINAL LEMMA':<18} | {'POS':<8} | PERSON")
    print("-" * 100)

    for _, r in rows.head(100).iterrows():
        print(
            f"{r['token']:<18} | "
            f"{r['raw_lemma']:<18} | "
            f"{r['final_lemma']:<18} | "
            f"{r['pos']:<8} | "
            f"{'YES' if r['is_person'] else 'NO'}"
        )

    print("\nPERSON TOKENS")
    persons = rows[rows["is_person"]]["token"].tolist()
    print(persons if persons else "None")

    print("\nVERB / AUX LEMMAS")
    verbs = rows[rows["pos"].isin(["VERB", "AUX"])]
    print(
        verbs[["token", "raw_lemma", "final_lemma", "pos"]]
        .drop_duplicates()
        .head(100)
        .to_string(index=False)
    )

    print()

DOCUMENT: HAM2-811011-007

TEXT SAMPLE
----------------------------------------------------------------------------------------------------
سمیه نصیری ها
جودی ابوت را سال ها است که می‌شناسم، دقیقا از دوران نوجوانی. شاید در دوران پرفراز
و نشیب نوجوانی دلم می‌خواست جای او باشم. خیلی از لحظاتی که ماجراهایش را می‌خواندم 
دوست داشتم دردسرهای او مال من باشد. جودی دختری است که اصولا دردسر درست می‌کند و
اتفاقا دردسرهایش اطرافیان را هم به دردسر می‌اندازد. خوب است قدری باانصاف باشیم. 
شاید از این جهت دلم می‌خواست جای جودی باشم که خداوند در تمام لحظات زندگی یار و یاورش 
بود. در تمام لحظات سیاه و بغرنج زندگی خداوند به یاری اش می‌آمد و اوضاع اش سروسامان 
پیدا می‌کرد. لطف خداوند همیشه شامل حال او بود. 
اما کاش جودی آنقدر خوش شانس نبود، همین قدر که تو در یک نوانخانه سیاه، کثیف، بدترکیب
روزگار تلخ و سیاهی را بگذرانی و از گرسنگی شکمت به پشتت چسبیده باشد و مثل دخترک 
کبریت فروش همه خواسته هایت را در حد آرزو ببینی، آن وقت سایه ای به اسم بابالنگ دراز
پیدا شود و تو را از آن نوانخانه اسفناک نجات دهد و به شب

In [24]:
person_rows = inspection_df[inspection_df["is_person"]]

person_docs = person_rows["doc_id"].drop_duplicates()

print(f"Documents with PERSON: {len(person_docs)}")
print(f"PERSON tokens       : {len(person_rows):,}")
print(f"Unique PERSON forms : {person_rows['token'].nunique():,}")

print("\nSample detected PERSON tokens:")
print(
    person_rows[["doc_id", "sent_idx", "token", "final_lemma", "pos"]]
    .head(100)
    .to_string(index=False)
)

print("\nPERSON tokens by document:")
for doc_id in person_docs.head(10):
    tokens = person_rows.loc[
        person_rows["doc_id"] == doc_id, "token"
    ].tolist()
    print(f"{doc_id}: {tokens}")

Documents with PERSON: 41
PERSON tokens       : 685
Unique PERSON forms : 336

Sample detected PERSON tokens:
         doc_id  sent_idx     token final_lemma   pos
HAM2-811011-007         0      سمیه        سمیه PROPN
HAM2-811011-007         0     نصیری       نصیری PROPN
HAM2-811011-007         0      جودی        جودی PROPN
HAM2-811011-007         0      ابوت        ابوت PROPN
HAM2-811011-007         3      جودی        جودی PROPN
HAM2-811011-007         8      جودی        جودی PROPN
HAM2-811011-007         8     آنقدر       آنقدر  NOUN
HAM2-811011-007         8   بابالنگ     بابالنگ  NOUN
HAM2-811011-007         8      جودی        جودی PROPN
HAM2-811011-007         8      ابوت        ابوت PROPN
HAM2-811011-007         9      جودی        جودی PROPN
HAM2-811011-007        11      جودی        جودی PROPN
HAM2-811011-007        12     جرویس       جرویس PROPN
HAM2-811011-007        12   پندلتون     پندلتون PROPN
HAM2-811011-007        12      جودی        جودی  NOUN
HAM2-811011-007        14 

In [25]:
suspect_tokens = {
    "ند", "ای", "آنقدر",
    "آمدن","محمدو"
}

for _, row in sample_docs.iterrows():
    text = row["norm_body"]
    if not isinstance(text, str):
        continue

    for sent_idx, sentence in enumerate(split_persian_sentences(text)):
        doc, flags, _ = tag_sentence(sentence)

        words = []
        flag_index = 0

        for s in doc.sentences:
            for w in s.words:
                words.append({
                    "text": w.text or "",
                    "lemma": w.lemma or "",
                    "pos": w.upos or "X",
                    "person": bool(flags[flag_index]),
                })
                flag_index += 1

        for i, w in enumerate(words):
            if w["text"] not in suspect_tokens or not w["person"]:
                continue

            start = max(0, i - 5)
            end = min(len(words), i + 6)

            print("\n" + "=" * 100)
            print(f"DOCUMENT : {row['doc_id']}")
            print(f"SENTENCE : {sent_idx}")

            print("\nRAW SENTENCE:")
            print(sentence)

            print("\nTOKEN CONTEXT:")
            for j in range(start, end):
                x = words[j]
                marker = " <<< SUSPECT PERSON" if j == i else ""

                print(
                    f"{j:3d} | "
                    f"{x['text']:<15} | "
                    f"lemma={x['lemma']:<15} | "
                    f"POS={x['pos']:<8} | "
                    f"PERSON={x['person']}{marker}"
                )


DOCUMENT : HAM2-811011-007
SENTENCE : 8

RAW SENTENCE:
اما کاش جودی آنقدر خوش شانس نبود، همین قدر که تو در یک نوانخانه سیاه، کثیف، بدترکیب روزگار تلخ و سیاهی را بگذرانی و از گرسنگی شکمت به پشتت چسبیده باشد و مثل دخترک  کبریت فروش همه خواسته هایت را در حد آرزو ببینی، آن وقت سایه ای به اسم بابالنگ دراز پیدا شود و تو را از آن نوانخانه اسفناک نجات دهد و به شبانه روزی بفرستد، می‌تواند نهایت شانس یک موجود باشد جودی ابوت با تمام توانایی هایش می‌توانست ادامه قصه اش را خودش پیش ببرد لااقل آنقدر متکی به امداد و شانس نباشد.

TOKEN CONTEXT:
  0 | اما             | lemma=اما             | POS=CCONJ    | PERSON=False
  1 | کاش             | lemma=کاش             | POS=INTJ     | PERSON=False
  2 | جودی            | lemma=جودی            | POS=PROPN    | PERSON=True
  3 | آنقدر           | lemma=آنقدر           | POS=NOUN     | PERSON=True <<< SUSPECT PERSON
  4 | خوش             | lemma=خوش             | POS=ADJ      | PERSON=False
  5 | شانس            | lemma=شانس            | POS=NOUN     | PERS

### Stopword, clitic, and copula filtering

Filtering is applied **after** tagging so the taggers receive complete sentence context. Removed:
function words/stopwords, clitics and copular elements that attach orthographically but carry no
lexical content, and tokens inside punctuation-delimited chunks. This happens before candidate
extraction so only content words enter the bigram counters.

In [26]:
STOPWORDS = {
    "و", "در", "به", "از", "را", "که", "این", "آن", "با", "برای", "تا", "یا",
    "هم", "نیز", "اما", "ولی", "چون", "اگر", "بر", "بی", "پس", "چه", "چرا",
    "کدام", "هر", "همه", "بعضی", "یک", "دو", "چند", "خود", "همچنین", "دیگر",
    "باید", "میان", "روی", "زیر", "بالای", "کنار", "طی", "بین", "علیه",
    "درباره", "توسط", "بجای", "مانند", "مثل", "طبق", "طریق",
    "ها", "های", "تر", "ترین", "ای", "وی", "‌ها", "‌های"
}

COPULA_LEMMAS = {"است", "بودن", "هست", "نیست", "شدن", "بود"}
PERSIAN_RE = re.compile(r"[\u0600-\u06FF]")

def is_content_token(text, lemma, pos, is_entity=False):
    text, lemma = text.strip(), lemma.strip()

    if not PERSIAN_RE.search(lemma) or len(lemma) < 3:
        return False

    if pos in {"PUNCT", "SYM", "NUM", "X"}:
        return False

    if lemma in STOPWORDS or text in STOPWORDS:
        return False

    if lemma in COPULA_LEMMAS or lemma in {"می", "نمی", "‌می", "‌نمی"}:
        return False

    if is_entity and pos == "PROPN":
        return False

    return True


def filter_tokens(tokens):
    out = []

    for text, lemma, pos, is_entity in tokens:
        lemma = normalize_verb_lemma(lemma, pos=pos, surface=text)

        if is_content_token(text, lemma, pos, is_entity):
            out.append((lemma, pos, is_entity))

    return out

### End-to-end check on the same 50 documents

Verify sentence boundaries, PERSON filtering, and candidate bigrams before starting full-corpus processing.

In [27]:
from collections import Counter
from tqdm.notebook import tqdm

CLAUSE_BOUNDARY_TEXT = {".", "!", "؟", "?", "؛", ";", ":", ",", "،"}
CLAUSE_BOUNDARY_POS = {"PUNCT", "SYM"}

VALID_POS_PATTERNS = {
    ("NOUN", "VERB"), ("NOUN", "NOUN"), ("NOUN", "ADJ"),
    ("ADJ", "NOUN"), ("ADJ", "VERB"),
    ("NOUN", "PROPN"), ("ADJ", "PROPN"),
    ("PROPN", "NOUN"), ("PROPN", "VERB"), ("PROPN", "ADJ"),
}

def stanza_words(doc, flags):
    words, i = [], 0

    for sent in doc.sentences:
        for word in sent.words:
            words.append((
                word.text or "",
                word.lemma or word.text or "",
                word.upos or "X",
                bool(flags[i])
            ))
            i += 1

    return words


def split_into_chunks(tokens):
    chunks, current = [], []

    for token in tokens:
        text, _, pos, _ = token

        if text.strip() in CLAUSE_BOUNDARY_TEXT or pos in CLAUSE_BOUNDARY_POS:
            if current:
                chunks.append(filter_tokens(current))
                current = []
        else:
            current.append(token)

    if current:
        chunks.append(filter_tokens(current))

    return chunks


def extract_pairs(tokens, window, pair_counter, pair_docs,
                   pos_counter, examples, unigram_counter, doc_id):

    for chunk in split_into_chunks(tokens):

        for word, pos, _ in chunk:
            unigram_counter[word] += 1

        for i, (w1, pos1, _) in enumerate(chunk):
            for w2, pos2, _ in chunk[i + 1:i + 1 + window]:

                if w1 == w2 or (pos1, pos2) not in VALID_POS_PATTERNS:
                    continue

                pair = (w1, w2)
                pair_counter[pair] += 1
                pair_docs.setdefault(pair, set()).add(doc_id)
                pos_counter[pair] = f"{pos1}+{pos2}"
                examples.setdefault(pair, " ".join(w for w, _, _ in chunk))


sample_pairs = Counter()
sample_unigrams = Counter()
sample_pos_pattern_counter = {}
sample_pair_docs = {}
sample_example_sentence = {}
sample_tagged_docs = []

window = CONFIG["WINDOW_SIZE"]

for _, row in tqdm(
    sample_docs.iterrows(),
    total=len(sample_docs),
    desc="Extracting Collocation Pairs"
):
    text, doc_id = row["norm_body"], row["doc_id"]

    if not isinstance(text, str) or not text.strip():
        continue

    tagged_sentences = []

    for sent_id, sentence in enumerate(split_persian_sentences(text)):
        doc, flags, _ = tag_sentence(sentence)
        tokens = stanza_words(doc, flags)

        tagged_sentences.append((sent_id, sentence, tokens))

        extract_pairs(
            tokens, window,
            sample_pairs,
            sample_pair_docs,
            sample_pos_pattern_counter,
            sample_example_sentence,
            sample_unigrams,
            doc_id
        )

    sample_tagged_docs.append((doc_id, tagged_sentences))

print(f"Distinct candidate pairs: {len(sample_pairs)}")

for (w1, w2), count in sample_pairs.most_common(20):
    print(
        f"{count:4d} | {w1:<15} + {w2:<15} | "
        f"{sample_pos_pattern_counter[(w1, w2)]}"
    )

Extracting Collocation Pairs:   0%|          | 0/50 [00:00<?, ?it/s]

Distinct candidate pairs: 21771
  28 | وجود            + داشتن           | NOUN+VERB
  21 | نرم             + افزار           | NOUN+NOUN
  17 | کار             + کردن            | NOUN+NOUN
  16 | استفاده         + کردن            | NOUN+VERB
  15 | پیدا            + کردن            | ADJ+VERB
  15 | قرار            + گرفتن           | NOUN+VERB
  15 | سرمایه          + گذاری           | NOUN+ADJ
  14 | نشان            + دادن            | NOUN+VERB
  14 | قرار            + دادن            | NOUN+VERB
  14 | سال             + آینده           | NOUN+ADJ
  14 | دوم             + خرداد           | ADJ+PROPN
  13 | زندگی           + کردن            | NOUN+VERB
  13 | انجام           + دادن            | NOUN+VERB
  12 | اظهار           + داشتن           | NOUN+VERB
  12 | صورت            + گرفتن           | NOUN+VERB
  12 | قرار            + داشتن           | NOUN+VERB
  12 | اجرا            + طرح             | NOUN+NOUN
  11 | احداث           + مترو            | NOUN+NOUN
  10 | افزایش    

In [28]:
from collections import defaultdict, Counter

surface_by_lemma = defaultdict(Counter)

for _, tagged_sentences in sample_tagged_docs:
    for _, _, tokens in tagged_sentences:
        for text, raw_lemma, pos, _ in tokens:
            if pos in {"PUNCT", "SYM", "NUM", "SPACE"} or not text.strip():
                continue

            surface = text.strip()
            lemma = normalize_verb_lemma(raw_lemma, pos=pos, surface=surface)
            surface_by_lemma[lemma][surface] += 1

target_lemmas = ["گرفتن", "دادن", "کردن", "داشتن", "شدن"]

print("=== VERB LEMMA → SURFACE FORMS ===\n")

for lemma in target_lemmas:
    forms = surface_by_lemma.get(lemma)

    if not forms:
        print(f"'{lemma}' → NOT FOUND\n{'-' * 50}")
        continue

    print(f"'{lemma}' → {sum(forms.values())} tokens, {len(forms)} forms")

    for surface, count in forms.most_common(20):
        print(f"  {surface:<15} x{count}")

    print("-" * 50)

=== VERB LEMMA → SURFACE FORMS ===

'گرفتن' → 100 tokens, 19 forms
  گرفته           x31
  می‌گیرد         x11
  گرفت            x10
  گرفتن           x7
  گیرد            x6
  بگیرد           x5
  می‌گیریم        x5
  می‌گیرند        x5
  بگیرید          x5
  گیرند           x3
  بگیریم          x2
  گرفتند          x2
  گرفتید          x2
  می‌گرفت         x1
  گرفتیم          x1
  می‌گیری         x1
  بگیرند          x1
  نگیرد           x1
  فراگرفت         x1
--------------------------------------------------
'دادن' → 147 tokens, 23 forms
  داده            x33
  می‌دهد          x22
  داد             x22
  دهد             x12
  دادن            x8
  می‌دهند         x8
  دهند            x7
  دهید            x6
  بدهد            x5
  بدهید           x5
  نداد            x3
  ندهید           x2
  می‌دادند        x2
  دهم             x2
  نمی‌دهیم        x2
  می‌دهیم         x1
  دادند           x1
  نداده           x1
  نمی‌دهد         x1
  دادیم           x1
--------------------------

In [29]:
from collections import Counter

person_counts = Counter()
entity_rows = []

for doc_id, tagged_sentences in sample_tagged_docs:
    for sent_id, sentence, tokens in tagged_sentences:
        for text, lemma, pos, is_person in tokens:
            if is_person:
                person_counts[text.strip()] += 1
                entity_rows.append(
                    (doc_id, sent_id, text.strip(), lemma, pos)
                )

print(f"=== NER REPORT ({len(sample_tagged_docs)} Documents) ===")
print(f"PERSON tokens : {sum(person_counts.values())}")
print(f"Unique tokens : {len(person_counts)}")

print("\n--- Most Frequent PERSON Tokens ---")
for word, count in person_counts.most_common(20):
    print(f"  {word:<18} x{count}")

print("\n--- Sample PERSON Detections ---")
for doc_id, sent_id, text, lemma, pos in entity_rows[:30]:
    print(
        f"  [PER] {text:<16} "
        f"lemma={lemma:<16} POS={pos:<7} "
        f"doc={doc_id}, sent={sent_id}"
    )

=== NER REPORT (50 Documents) ===
PERSON tokens : 685
Unique tokens : 336

--- Most Frequent PERSON Tokens ---
  جودی               x23
  سید                x15
  رضا                x12
  زاده               x11
  علی                x10
  سیدحسینی           x10
  حسینی              x10
  محمد               x9
  محمود              x8
  هاشمی              x7
  ادموند             x7
  جرویس              x6
  بزیک               x6
  تقی                x6
  توکل               x6
  ابوت               x5
  پندلتون            x5
  عباس               x5
  الدین              x5
  عبدالله            x5

--- Sample PERSON Detections ---
  [PER] سمیه             lemma=سمیه             POS=PROPN   doc=HAM2-811011-007, sent=0
  [PER] نصیری            lemma=نصیری            POS=PROPN   doc=HAM2-811011-007, sent=0
  [PER] جودی             lemma=جودی             POS=PROPN   doc=HAM2-811011-007, sent=0
  [PER] ابوت             lemma=ابوت             POS=PROPN   doc=HAM2-811011-007, sent=0
  [PER] جودی    

In [30]:
import random

print("=== POSSIBLE MWT / TOKENIZATION ISSUES ===\n")

MAX_SAMPLES = 20
results = []
total_checked = 0

for doc_id, tagged_sentences in sample_tagged_docs:
    for sent_id, sentence, tokens in tagged_sentences:
        for i in range(len(tokens) - 1):
            t1, _, _, _ = tokens[i]
            t2, _, _, _ = tokens[i + 1]
            total_checked += 1
            
            if (
                t1.strip()
                and t2.strip()
                and len(t2.strip()) <= 2
                and all("\u0600" <= c <= "\u06FF" or c == "\u200c" for c in t2.strip())
            ):
                if len(results) < MAX_SAMPLES:
                    results.append((doc_id, sent_id, sentence, t1, t2))
                else:
                    j = random.randint(0, len(results))
                    if j < MAX_SAMPLES:
                        results[j] = (doc_id, sent_id, sentence, t1, t2)

for doc_id, sent_id, sentence, t1, t2 in results:
    print(f"DOCUMENT : {doc_id}")
    print(f"SENTENCE : {sent_id}")
    print(f"RAW      : {sentence}")
    print(f"TOKENS   : '{t1}' + '{t2}'")
    print("-" * 80)

=== POSSIBLE MWT / TOKENIZATION ISSUES ===

DOCUMENT : HAM2-811025-098
SENTENCE : 2
RAW      : طالقانی با تاکید مجدد بر این مطلب که همانند کمیته داوران، کمیته مربیان نیز در فدراسیون کشتی تشکیل خواهد شد اظهار داشت: در جهت بازبینی کار مربیان و کاهش اشتباهات،  همانند کمیته داوران، از سال آینده کمیته مربیان نیز به طور رسمی در فدراسیون تشکیل  خواهد شد و کلیه مربیان از سوی این کمیته انتخاب و به کار مربیگری خواهند پرداخت.
TOKENS   : 'شد' + 'و'
--------------------------------------------------------------------------------
DOCUMENT : HAM2-811025-098
SENTENCE : 2
RAW      : طالقانی با تاکید مجدد بر این مطلب که همانند کمیته داوران، کمیته مربیان نیز در فدراسیون کشتی تشکیل خواهد شد اظهار داشت: در جهت بازبینی کار مربیان و کاهش اشتباهات،  همانند کمیته داوران، از سال آینده کمیته مربیان نیز به طور رسمی در فدراسیون تشکیل  خواهد شد و کلیه مربیان از سوی این کمیته انتخاب و به کار مربیگری خواهند پرداخت.
TOKENS   : 'داوران' + '،'
-----------------------------------------------------------------------------

In [31]:
import random

print("=== SHORT TOKEN AUDIT ===\n")

MAX_SAMPLES = 20
results = []

for doc_id, tagged_sentences in sample_tagged_docs:
    for sent_id, sentence, tokens in tagged_sentences:
        for i, (text, lemma, pos, is_person) in enumerate(tokens):
            text = text.strip()
            
            if not text or len(text) > 2 or pos in {"PUNCT", "SYM"}:
                continue
            
            left = tokens[max(0, i - 2):i]
            right = tokens[i + 1:i + 3]
            
            left_text = " ".join(t[0] for t in left)
            right_text = " ".join(t[0] for t in right)
            
            item = (doc_id, sent_id, text, lemma, pos, is_person, left_text, right_text)
            
            if len(results) < MAX_SAMPLES:
                results.append(item)
            else:
                j = random.randint(0, len(results))
                if j < MAX_SAMPLES:
                    results[j] = item

for doc_id, sent_id, text, lemma, pos, is_person, left_text, right_text in results:
    print(
        f"{doc_id} | sent={sent_id} | "
        f"'{text}' | lemma={lemma} | POS={pos} | "
        f"NER={is_person}"
    )
    print(f"  Context: {left_text} >>> [{text}] <<< {right_text}")
    print()

=== SHORT TOKEN AUDIT ===

HAM2-811025-098 | sent=3 | 'و' | lemma=و | POS=CCONJ | NER=False
  Context: المپیک شویم >>> [و] <<< در تمام

HAM2-811025-098 | sent=2 | 'شد' | lemma=شد | POS=VERB | NER=False
  Context: تشکیل خواهد >>> [شد] <<< و کلیه

HAM2-811025-098 | sent=2 | 'و' | lemma=و | POS=CCONJ | NER=False
  Context: خواهد شد >>> [و] <<< کلیه مربیان

HAM2-811025-098 | sent=3 | 'از' | lemma=از | POS=ADP | NER=False
  Context: جانبه ای >>> [از] <<< فدراسیون کشتی

HAM2-811025-098 | sent=3 | 'در' | lemma=در | POS=ADP | NER=False
  Context: بدهد و >>> [در] <<< این مسیر

HAM2-811025-098 | sent=2 | 'بر' | lemma=بر | POS=ADP | NER=False
  Context: تاکید مجدد >>> [بر] <<< این مطلب

HAM2-811025-098 | sent=3 | 'ها' | lemma=ها | POS=NOUN | NER=False
  Context: تمام درخواست >>> [ها] <<< و برنامه

HAM2-811025-098 | sent=2 | 'در' | lemma=در | POS=ADP | NER=False
  Context: طور رسمی >>> [در] <<< فدراسیون تشکیل

HAM2-811025-098 | sent=3 | 'وی' | lemma=وی | POS=PRON | NER=False
  Context:  >>> [وی] <

In [32]:
from collections import Counter

filter_stats = Counter()
removed_examples = Counter()

for _, tagged_sentences in sample_tagged_docs:
    for _, _, tokens in tagged_sentences:
        for text, raw_lemma, pos, is_person in tokens:

            text = text.strip()
            lemma = normalize_verb_lemma(
                raw_lemma, pos=pos, surface=text
            )

            if not text:
                filter_stats["empty"] += 1
            elif pos in {"PUNCT", "SYM", "NUM", "X"}:
                filter_stats["POS"] += 1
            elif len(lemma) < 3:
                filter_stats["short_<3"] += 1
                removed_examples[lemma] += 1
            elif lemma in STOPWORDS or text in STOPWORDS:
                filter_stats["stopword"] += 1
                removed_examples[lemma] += 1
            elif lemma in COPULA_LEMMAS:
                filter_stats["copula"] += 1
                removed_examples[lemma] += 1
            elif is_person and pos == "PROPN":
                filter_stats["NER+PROPN"] += 1
                removed_examples[lemma] += 1
            else:
                filter_stats["kept"] += 1

print("=== FILTERING AUDIT ===")

for key, count in filter_stats.items():
    print(f"{key:<15} : {count}")

print("\n--- Most Frequent Removed Tokens ---")

for word, count in removed_examples.most_common(30):
    print(f"{word:<20} x{count}")

=== FILTERING AUDIT ===
NER+PROPN       : 634
short_<3        : 8897
kept            : 20370
copula          : 1700
POS             : 3347
stopword        : 2152

--- Most Frequent Removed Tokens ---
و                    x1603
بودن                 x1177
در                   x1099
به                   x1045
از                   x880
که                   x800
این                  x559
شدن                  x544
را                   x532
های                  x362
با                   x336
آن                   x254
برای                 x218
ها                   x197
خود                  x188
او                   x156
یا                   x133
هم                   x127
ای                   x124
بر                   x113
آب                   x110
ما                   x107
تا                   x92
هر                   x89
دیگر                 x86
نیز                  x80
اما                  x77
بایست                x69
اگر                  x66
همه                  x54


**Gate before continuing:** the sample should show clean sentence boundaries, valid content-word bigrams, and PERSON entities aligned to the correct tokens. The verb mapping should collapse inflected forms consistently. If the sample fails these checks, stop before the full-corpus run.

## 7. Full Corpus Processing (Resumable)

Process every document with the same sentence segmentation, Stanza tagging, PERSON NER, and token
alignment validated on the 50-document sample.

Results are appended to **`sentences_raw.jsonl`** — one JSON record per tagged sentence plus a
`doc_done` marker per document. On restart, already-completed documents are read from the checkpoint
and skipped, making the run fully **resumable**. Checkpointing is essential for a corpus of this size:
a crash or session timeout (e.g., on Kaggle) must not discard hours of GPU work, and writes are
flushed periodically so at most a small batch is ever lost.

In [40]:
sentences_path = Path(CONFIG["CHECKPOINT_DIR"]) / "sentences_raw.jsonl"
done = set()

if sentences_path.exists():
    with open(sentences_path, encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
                if r.get("type") == "doc_done":
                    done.add(r["doc_id"])
            except Exception:
                pass

full_corpus_df = (
    corpus_df.sample(n=CONFIG["N_DOCS_SUBSET"], random_state=CONFIG["SEED"]).reset_index(drop=True)
    if CONFIG["N_DOCS_SUBSET"] else corpus_df
)

remaining = full_corpus_df[~full_corpus_df["doc_id"].isin(done)]
print(f"Done: {len(done):,} | Remaining: {len(remaining):,} / {len(full_corpus_df):,}")

t0 = time.time()

with open(sentences_path, "a", encoding="utf-8") as f:
    for n, (_, row) in enumerate(remaining.iterrows(), 1):
        doc_id, text = row["doc_id"], row["norm_body"]

        if isinstance(text, str) and text.strip():
            for sent_id, sentence in enumerate(split_persian_sentences(text)):
                doc, flags, _ = tag_sentence(sentence)
                tokens = stanza_words(doc, flags)
                f.write(json.dumps({
                    "type": "sentence",
                    "doc_id": doc_id,
                    "sent_id": sent_id,
                    "sentence": sentence,
                    "tokens": tokens
                }, ensure_ascii=False) + "\n")

        f.write(json.dumps({"type": "doc_done", "doc_id": doc_id}) + "\n")

        if n % 200 == 0:
            f.flush()
            os.fsync(f.fileno())
            elapsed = time.time() - t0
            rate = n / elapsed
            eta = (len(remaining) - n) / rate / 60
            print(f"[{n:,}/{len(remaining):,}] {rate:.2f} docs/s | ETA {eta:.1f} min")

    f.flush()
    os.fsync(f.fileno())

print("Sentence/tokenization checkpoint complete.")

Done: 4,714 | Remaining: 661 / 5,375
[200/661] 0.82 docs/s | ETA 9.3 min
[400/661] 0.83 docs/s | ETA 5.3 min
[600/661] 0.83 docs/s | ETA 1.2 min
Sentence/tokenization checkpoint complete.


In [41]:
n_sentences = n_raw = n_kept = 0

with open(sentences_path, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        if rec.get("type") != "sentence":
            continue

        n_sentences += 1
        tokens = [
            (t[0], t[1], t[2], bool(t[3]))
            for t in rec["tokens"]
        ]
        n_raw += len(tokens)

        if len(filter_tokens(tokens)) >= 2:
            n_kept += 1

print(f"Sentence records : {n_sentences:,}")
print(f"Raw tokens       : {n_raw:,}")
print(f"Usable sentences : {n_kept:,}")

Sentence records : 93,455
Raw tokens       : 3,124,745
Usable sentences : 92,347


## 8. Candidate Collocation Extraction

Candidate bigrams are extracted with a **window-based** scan over each sentence chunk (default
`WINDOW_SIZE = 2`, i.e. adjacent content words):

- Candidates never cross sentence or punctuation-chunk boundaries.
- Only pairs matching a **valid POS pattern** (e.g. `NOUN+NOUN`, `ADJ+NOUN`, `NOUN+VERB`) are kept.
- For each pair we accumulate **corpus frequency** (total occurrences) and **document frequency**
  (number of distinct documents containing the pair) — the latter guards against pairs repeated
  within a single article dominating the rankings.

In [ ]:
def extract_pairs_from_sentence(tokens, window, pair_counter=None, pair_docs=None, pos_pattern_counter=None, example_sentence=None,
                                unigram_counter=None, doc_id=None):
    
    found_pairs = set()

    for chunk in split_into_chunks(tokens):
        if unigram_counter is not None:
            for word, pos, _ in chunk:
                unigram_counter[word] += 1

        for i, (w1, pos1, _) in enumerate(chunk):
            for w2, pos2, _ in chunk[i + 1:i + 1 + window]:
                if w1 == w2 or (pos1, pos2) not in VALID_POS_PATTERNS:
                    continue

                pair = (w1, w2)
                found_pairs.add(pair)

                if pair_counter is not None:
                    pair_counter[pair] += 1
                if pair_docs is not None:
                    pair_docs.setdefault(pair, set()).add(doc_id)
                if pos_pattern_counter is not None:
                    pos_pattern_counter[pair] = f"{pos1}+{pos2}"
                if example_sentence is not None:
                    example_sentence.setdefault(pair, " ".join(w for w, _, _ in chunk))

    return found_pairs


CANDIDATE_STATE = Path(CONFIG["CHECKPOINT_DIR"]) / "candidates_state.pkl"
BATCH_SIZE = 200
window = CONFIG["WINDOW_SIZE"]

if CANDIDATE_STATE.exists():
    with open(CANDIDATE_STATE, "rb") as f:
        state = pickle.load(f)
    print(f"Resumed: {len(state['done_docs']):,} documents")
else:
    state = {
        "done_docs": set(),
        "pair_counter": Counter(),
        "unigram_counter": Counter(),
        "pair_doc_freq": Counter(),
        "pair_pos": defaultdict(Counter),
    }

docs_since_ckpt = 0
doc_pairs = set()
current_doc = None
t0 = time.time()

with open(sentences_path, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)

        if rec.get("type") == "sentence":
            doc_id = rec["doc_id"]

            if doc_id in state["done_docs"]:
                continue

            tokens = [
                (t[0], t[1], t[2], bool(t[3]))
                for t in rec["tokens"]
            ]

            if current_doc != doc_id:
                current_doc = doc_id
                doc_pairs = set()

            local_pairs = extract_pairs_from_sentence(
                tokens,
                window,
                pair_counter=state["pair_counter"],
                pair_docs=None,
                pos_pattern_counter=None,
                example_sentence=None,
                unigram_counter=state["unigram_counter"],
                doc_id=doc_id,
            )

            doc_pairs.update(local_pairs)

        elif rec.get("type") == "doc_done":
            doc_id = rec["doc_id"]

            if doc_id in state["done_docs"]:
                continue

            state["pair_doc_freq"].update(doc_pairs)
            state["done_docs"].add(doc_id)

            docs_since_ckpt += 1
            current_doc = None
            doc_pairs = set()

            if docs_since_ckpt % BATCH_SIZE == 0:
                with open(CANDIDATE_STATE, "wb") as f:
                    pickle.dump(state, f, protocol=pickle.HIGHEST_PROTOCOL)

                elapsed = time.time() - t0
                print(
                    f"[CHECKPOINT] docs={len(state['done_docs']):,} "
                    f"| pairs={len(state['pair_counter']):,} "
                    f"| unigrams={len(state['unigram_counter']):,} "
                    f"| time={elapsed/60:.1f} min"
                )

with open(CANDIDATE_STATE, "wb") as f:
    pickle.dump(state, f, protocol=pickle.HIGHEST_PROTOCOL)

print("\nCandidate extraction complete.")
print(f"Documents : {len(state['done_docs']):,}")
print(f"Pairs     : {len(state['pair_counter']):,}")
print(f"Lemmas    : {len(state['unigram_counter']):,}")

[CHECKPOINT] docs=200 | pairs=82,718 | unigrams=8,150 | time=0.0 min
[CHECKPOINT] docs=400 | pairs=150,904 | unigrams=11,048 | time=0.0 min
[CHECKPOINT] docs=600 | pairs=209,130 | unigrams=13,270 | time=0.0 min
[CHECKPOINT] docs=800 | pairs=269,085 | unigrams=15,630 | time=0.0 min
[CHECKPOINT] docs=1,000 | pairs=318,791 | unigrams=17,406 | time=0.0 min
[CHECKPOINT] docs=1,200 | pairs=364,917 | unigrams=18,947 | time=0.1 min
[CHECKPOINT] docs=1,400 | pairs=405,317 | unigrams=20,256 | time=0.1 min
[CHECKPOINT] docs=1,600 | pairs=444,970 | unigrams=21,586 | time=0.1 min
[CHECKPOINT] docs=1,800 | pairs=482,806 | unigrams=22,780 | time=0.1 min
[CHECKPOINT] docs=2,000 | pairs=521,032 | unigrams=23,906 | time=0.1 min
[CHECKPOINT] docs=2,200 | pairs=555,516 | unigrams=24,791 | time=0.1 min
[CHECKPOINT] docs=2,400 | pairs=592,743 | unigrams=25,768 | time=0.1 min
[CHECKPOINT] docs=2,600 | pairs=629,927 | unigrams=26,711 | time=0.1 min
[CHECKPOINT] docs=2,800 | pairs=668,846 | unigrams=27,623 | t

## 9. Association Measures

Raw frequency confounds collocation strength with word commonness. Four standard association measures
are computed from the same adjacent-content-word event definition used during extraction, each capturing
a different aspect of collocation strength:

| Measure | What it captures |
|---|---|
| **PMI** | Pointwise Mutual Information — how much more often the pair occurs than independent occurrence would predict; favors rare-but-exclusive pairs. |
| **t-score** | Statistical confidence that the co-occurrence exceeds chance; more robust for lower-frequency pairs than PMI. |
| **logDice** | Symmetric, frequency-balanced Dice-style measure; robust and widely used in lexicography. |
| **LLR** | Log-Likelihood Ratio — significance of the association, well behaved across the whole frequency spectrum. |

No single measure is sufficient: PMI overrates rare pairs, t-score underrates them, logDice balances
frequency, and LLR gives global statistical significance. Combining them yields a more stable ranking.

In [ ]:
candidates_df = pd.DataFrame(rows)

candidates_df.to_parquet(CKPT_CANDIDATES_RAW, index=False)

print("\n" + "=" * 100)
print("CANDIDATE EXTRACTION SUMMARY")
print("=" * 100)
print(f"Total candidate pairs : {len(candidates_df):,}")
print(f"Total pair events     : {TOTAL_PAIR_EVENTS:,}")
print(f"Unique lemmas         : {len(unigram_counter):,}")

display_cols = [
    "word1", "word2",
    "frequency", "doc_frequency",
    "pos_pattern",
    "pmi", "t_score", "logdice", "llr"
]

preview = (
    candidates_df[
        candidates_df["frequency"] >= CONFIG["MIN_FREQUENCY"]
    ]
    .sort_values(
        ["frequency", "doc_frequency"],
        ascending=[False, False]
    )
    .head(20)
    .copy()
)

preview["pmi"] = preview["pmi"].round(3)
preview["t_score"] = preview["t_score"].round(3)
preview["logdice"] = preview["logdice"].round(3)
preview["llr"] = preview["llr"].round(2)

print("\nTop 20 candidates by frequency:\n")

display(
    preview[display_cols].reset_index(drop=True)
)

print("\nExample sentences for top 10:\n")

for i, row in preview.head(10).iterrows():
    print(
        f"{i+1:02d}. {row['word1']} + {row['word2']} "
        f"| freq={row['frequency']:,} "
        f"| docs={row['doc_frequency']:,} "
        f"| POS={row['pos_pattern']}"
    )
    print(f"    {candidates_df.loc[i, 'example_sentence']}")
    print()


CANDIDATE EXTRACTION SUMMARY
Total candidate pairs : 1,071,546
Total pair events     : 2,827,269
Unique lemmas         : 36,981

Top 20 candidates by frequency:



,word1,word2,frequency,doc_frequency,pos_pattern,pmi,t_score,logdice,llr
0,قرار,گرفتن,2419,1483,NOUN+VERB,7.130,48.832,12.434,21065.72
1,وجود,داشتن,2388,1263,NOUN+VERB,6.136,48.173,11.531,17486.52
2,اعلام,کردن,1847,1209,NOUN+VERB,5.241,41.840,10.368,11186.77
3,نشان,دادن,1787,1053,NOUN+VERB,7.417,42.026,11.842,17344.69
4,قرار,دادن,1270,879,NOUN+VERB,5.537,34.870,11.066,7680.45
5,سال,گذشته,1218,762,NOUN+ADJ,6.507,34.516,11.351,9247.71
6,صورت,گرفتن,1135,790,NOUN+VERB,6.341,33.274,11.453,8189.78
7,پیدا,کردن,1105,735,ADJ+VERB,5.884,32.679,9.691,8439.35
8,حال,حاضر,1068,727,NOUN+ADJ,8.392,32.583,12.295,11401.08
9,سرمایه,گذاری,996,413,NOUN+ADJ,9.738,31.522,13.116,12867.24



Example sentences for top 10:

888. قرار + گرفتن | freq=2,419 | docs=1,483 | POS=NOUN+VERB
    ظرفیت بدن در مقابله با استرس می‌تواند تحت تاثیر عوامل گوناگونی قرار گیرند، چون:  وراثت، تجربه دوران کودکی، رژیم غذایی، نحوه خواب و ورزش، بود یا نبود روابط شخصی نزدیک،  سطح درآمد و موقعیت اجتماعی و نیز به واسطه انباشته کردن استرس های عادی تاحدی که بیش  از ظرفیت بدن باشند.

759. وجود + داشتن | freq=2,388 | docs=1,263 | POS=NOUN+VERB
    دو دهه قبل، بسیاری از دانشمندان به این عقیده که وضعیت ذهنی می‌تواند باعث بیماری شود، به  دیده تمسخر می‌نگریستند در آن زمان بین ذهن و بدن، مرز تیره ای وجود داشت که بهتر بود آن را به دست روانپزشکان بسپارند.

1387. اعلام + کردن | freq=1,847 | docs=1,209 | POS=NOUN+VERB
    سود دیگر تداوم ورزش، چنان که پژوهشگران اعلام کرده اند، افراد را از ابتلای به دیابت محفوظ نگه  می‌دارد، به شرطی که ورزش سریع باشد و باعث تعرق گردد.

723. نشان + دادن | freq=1,787 | docs=1,053 | POS=NOUN+VERB
    چندین دهه است که تحقیقات نشان داده اند استرس فیزیکی باعث آسیبرساندن به بدن  می‌شود، ا

## 10. Candidate Filtering

Candidates are filtered by two thresholds:

- **Minimum frequency** (`MIN_FREQUENCY`) — removes hapax-legomena and noise; association measures are
  unreliable for very low counts.
- **Minimum document frequency** (`MIN_DOC_FREQUENCY`) — requires the pair to appear in several distinct
  documents, suppressing article-specific repetition (e.g., a name repeated twenty times in one piece).

Using both thresholds together keeps pairs that are both frequent enough to score reliably **and**
distributed across the corpus rather than concentrated in a few documents.

In [46]:
CKPT_CANDIDATES_FILTERED = (
    Path(CONFIG["CHECKPOINT_DIR"]) / "candidates_filtered.parquet"
)

filtered_df = candidates_df[
    (candidates_df["frequency"] >= CONFIG["MIN_FREQUENCY"]) &
    (candidates_df["doc_frequency"] >= CONFIG["MIN_DOC_FREQUENCY"])
].reset_index(drop=True)

filtered_df.to_parquet(
    CKPT_CANDIDATES_FILTERED,
    index=False
)

print(f"Before: {len(candidates_df):,}")
print(f"After : {len(filtered_df):,}")
print(
    f"Frequency >= {CONFIG['MIN_FREQUENCY']}, "
    f"Doc frequency >= {CONFIG['MIN_DOC_FREQUENCY']}"
)

Before: 1,071,546
After : 22,120
Frequency >= 10, Doc frequency >= 3


## 11. POS Analysis — Majority-Vote POS Assignment

A pair may occur with different POS patterns in different contexts. For presentation, each word receives
the **most frequent (majority-vote) POS** observed for its lemma across the corpus, together with a
**confidence** value (share of the majority label). Low-confidence assignments are inspected below.

In [48]:
print("\nLow-confidence lemma POS assignments:")

display(
    lemma_pos_df[
        (lemma_pos_df["total_count"] >= 20) &
        (lemma_pos_df["pos_confidence"] < 0.80)
    ][
        ["lemma", "total_count", "majority_pos",
         "pos_confidence", "pos_distribution"]
    ]
    .sort_values(["pos_confidence", "total_count"])
    .head(30)
    .style
    .format({
        "total_count": "{:,}",
        "pos_confidence": "{:.2%}",
    })
)


Low-confidence lemma POS assignments:


,lemma,total_count,majority_pos,pos_confidence,pos_distribution
4812,اینگونه,172,ADJ,34.30%,"{'ADJ': 59, 'DET': 47, 'NOUN': 46, 'ADV': 20}"
14538,ترکمن,32,ADJ,34.38%,"{'ADJ': 11, 'PROPN': 11, 'NOUN': 10}"
1680,واحدی,75,NOUN,37.33%,"{'PROPN': 20, 'ADJ': 27, 'NOUN': 28}"
92,ای,"8,817",INTJ,37.72%,"{'INTJ': 3326, 'NOUN': 3155, 'ADJ': 1984, 'PROPN': 230, 'PRON': 48, 'CCONJ': 69, 'ADP': 4, 'AUX': 1}"
1415,اصفهانی,53,NOUN,37.74%,"{'PROPN': 16, 'ADJ': 17, 'NOUN': 20}"
5597,خبره,84,NOUN,39.29%,"{'ADJ': 31, 'NOUN': 33, 'PROPN': 20}"
9685,بعثی,33,ADJ,39.39%,"{'ADJ': 13, 'NOUN': 10, 'PROPN': 10}"
5173,تابعه,48,ADJ,39.58%,"{'NOUN': 15, 'PROPN': 14, 'ADJ': 19}"
10735,دانست,20,ADP,40.00%,"{'NOUN': 5, 'ADP': 8, 'PROPN': 2, 'AUX': 1, 'ADJ': 4}"
8084,بری,25,PROPN,40.00%,"{'ADJ': 9, 'NOUN': 6, 'PROPN': 10}"


## 12. Final Ranking — Weighted Combined Score

Final frequency thresholds are applied and candidates are ranked with a weighted combination of the four
association measures. Each metric is converted to a **percentile rank** (making the scales comparable)
and combined as:

```
combined = 0.30·PMI + 0.20·t-score + 0.30·logDice + 0.20·LLR   (weights sum to 1.0)
```

PMI and logDice receive higher weights because they best capture true collocational exclusivity and
frequency balance; t-score and LLR contribute statistical robustness. The top
`TOP_N_FINAL = 5000` pairs form the final dataset.

> Note: this cell uses percentile ranking. A separate Min-Max-scaled validation ranking is provided at
> the end of the notebook.

In [50]:
CKPT_BEST_DF = Path(CONFIG["CHECKPOINT_DIR"]) / "best_df_scored.parquet"
lemma_pos = pd.read_parquet(
    Path(CONFIG["CHECKPOINT_DIR"]) / "lemma_pos_majority.parquet"
).set_index("lemma")["majority_pos"].to_dict()

best_df = candidates_df[
    (candidates_df.frequency >= CONFIG["FINAL_MIN_FREQUENCY"]) &
    (candidates_df.doc_frequency >= CONFIG["FINAL_MIN_DOC_FREQUENCY"])
].copy()

def fix_pos(r):
    p1, p2 = r.pos_pattern.split("+")
    return f"{lemma_pos.get(r.word1,p1)}+{lemma_pos.get(r.word2,p2)}"

best_df["pos_pattern"] = best_df.apply(fix_pos, axis=1)

weights = {"pmi": .30, "t_score": .20, "logdice": .30, "llr": .20}
for m, w in weights.items():
    best_df[f"{m}_norm"] = best_df[m].rank(pct=True).fillna(0)

best_df["combined_score"] = sum(
    best_df[f"{m}_norm"] * w for m, w in weights.items()
)

best_df = best_df.nlargest(
    CONFIG["TOP_N_FINAL"], "combined_score"
).reset_index(drop=True)
best_df["pair_id"] = best_df.index + 1
best_df.to_parquet(CKPT_BEST_DF, index=False)

display(best_df[[
    "pair_id","word1","word2","frequency",
    "doc_frequency","pos_pattern","combined_score"
]].head(20).style.format({"combined_score": "{:.4f}"}))

,pair_id,word1,word2,frequency,doc_frequency,pos_pattern,combined_score
0,1,گفت,وگو,814,608,NOUN+NOUN,0.9944
1,2,دانش,آموزان,476,152,NOUN+NOUN,0.9938
2,3,جست,وجو,167,109,NOUN+NOUN,0.9937
3,4,نرم,افزار,184,71,NOUN+ADJ,0.9935
4,5,علاقه,مندان,198,147,NOUN+NOUN,0.9921
5,6,وسیله,نقلیه,240,100,NOUN+ADJ,0.9912
6,7,مرگ,میر,200,117,NOUN+NOUN,0.9909
7,8,صرفه,جوی,125,81,NOUN+NOUN,0.9907
8,9,بهره,بردار,508,269,NOUN+NOUN,0.9906
9,10,آیین,نامه,285,96,NOUN+NOUN,0.9904


## 13. Real Example Sentences

Each final collocation receives up to **`N_EXAMPLES_PER_PAIR = 5`** real sentences extracted from the
corpus. Sentences must contain the target pair under the same tagging/filtering pipeline and fall within
the configured length range.

Authentic corpus examples make the dataset immediately usable in end-user applications (dictionary
lookup, writing aids, language learning): users see the collocation in genuine journalistic context
rather than an abstract score alone.

In [56]:
CKPT_PAIR_EXAMPLES = Path(CONFIG["CHECKPOINT_DIR"]) / "pair_examples.json"
CKPT_EXAMPLES_PROGRESS = Path(CONFIG["CHECKPOINT_DIR"]) / "examples_progress.json"

target_pairs = set(zip(best_df.word1, best_df.word2))
N = CONFIG["N_EXAMPLES_PER_PAIR"]
MIN_W, MAX_W = CONFIG["EXAMPLE_MIN_WORDS"], CONFIG["EXAMPLE_MAX_WORDS"]
WINDOW = CONFIG["WINDOW_SIZE"]

if CKPT_PAIR_EXAMPLES.exists():
    with open(CKPT_PAIR_EXAMPLES, encoding="utf-8") as f:
        pair_examples = defaultdict(
            list, {tuple(k.split("␟")): v for k, v in json.load(f).items()}
        )
else:
    pair_examples = defaultdict(list)

done_docs = set()
if CKPT_EXAMPLES_PROGRESS.exists():
    with open(CKPT_EXAMPLES_PROGRESS, encoding="utf-8") as f:
        done_docs = set(json.load(f))

def get_pairs(tokens):
    found = set()

    for chunk in split_into_chunks(tokens):
        # chunk خروجی split_into_chunks سه‌تایی است
        ft = [
            (w, p, e) for w, p, e in chunk
            if p not in {"PUNCT", "SYM", "NUM", "SPACE", "X"} and not e
        ]

        for i, (w1, p1, e1) in enumerate(ft):
            for w2, p2, e2 in ft[i + 1:i + 1 + WINDOW]:
                if w1 != w2 and (p1, p2) in VALID_POS_PATTERNS:
                    found.add((w1, w2))

    return found

with open(sentences_path, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)

        if rec["type"] == "sentence":
            if rec["doc_id"] in done_docs:
                continue

            sentence = rec.get("sentence", "")
            if not MIN_W <= len(sentence.split()) <= MAX_W:
                continue

            tokens = [
                (t[0], t[1], t[2], bool(t[3]))
                for t in rec["tokens"]
            ]

            for pair in get_pairs(tokens) & target_pairs:
                examples = pair_examples[pair]

                if len(examples) < N and sentence not in {x["sentence"] for x in examples}:
                    examples.append({
                        "doc_id": rec["doc_id"],
                        "sent_id": rec["sent_id"],
                        "sentence": sentence
                    })

        elif rec["type"] == "doc_done":
            done_docs.add(rec["doc_id"])

            if len(done_docs) % 200 == 0:
                with open(CKPT_PAIR_EXAMPLES, "w", encoding="utf-8") as f:
                    json.dump({"␟".join(k): v for k, v in pair_examples.items()},
                              f, ensure_ascii=False)

                with open(CKPT_EXAMPLES_PROGRESS, "w", encoding="utf-8") as f:
                    json.dump(sorted(done_docs), f, ensure_ascii=False)

                full = sum(len(pair_examples[p]) >= N for p in target_pairs)
                print(f"{len(done_docs):,} docs | {full:,}/{len(target_pairs):,} complete")

with open(CKPT_PAIR_EXAMPLES, "w", encoding="utf-8") as f:
    json.dump({"␟".join(k): v for k, v in pair_examples.items()},
              f, ensure_ascii=False)

with open(CKPT_EXAMPLES_PROGRESS, "w", encoding="utf-8") as f:
    json.dump(sorted(done_docs), f, ensure_ascii=False)

print(
    f"Done: {len(done_docs):,} docs | "
    f"{sum(bool(pair_examples[p]) for p in target_pairs):,}/"
    f"{len(target_pairs):,} pairs have examples"
)

200 docs | 119/5,000 complete
400 docs | 291/5,000 complete
600 docs | 424/5,000 complete
800 docs | 597/5,000 complete
1,000 docs | 725/5,000 complete
1,200 docs | 880/5,000 complete
1,400 docs | 1,053/5,000 complete
1,600 docs | 1,160/5,000 complete
1,800 docs | 1,334/5,000 complete
2,000 docs | 1,509/5,000 complete
2,200 docs | 1,663/5,000 complete
2,400 docs | 1,802/5,000 complete
2,600 docs | 1,931/5,000 complete
2,800 docs | 2,071/5,000 complete
3,000 docs | 2,197/5,000 complete
3,200 docs | 2,315/5,000 complete
3,400 docs | 2,422/5,000 complete
3,600 docs | 2,540/5,000 complete
3,800 docs | 2,668/5,000 complete
4,000 docs | 2,770/5,000 complete
4,200 docs | 2,859/5,000 complete
4,400 docs | 2,971/5,000 complete
4,600 docs | 3,045/5,000 complete
4,800 docs | 3,163/5,000 complete
5,000 docs | 3,276/5,000 complete
5,200 docs | 3,365/5,000 complete
Done: 5,375 docs | 4,901/5,000 pairs have examples


If some pairs have fewer than the configured number of examples after the full scan, they are simply left with fewer examples. Adjust the example length range if broader coverage is required.

## 14. Final Dataset

The final dataset (`best_df.xlsx` / `best_df.parquet`) contains one row per collocation with:

| Field | Description |
|---|---|
| `pair_id` | Rank identifier (1..5000) |
| `word1`, `word2` | Canonical lemmas of the pair |
| `frequency` | Total corpus occurrences |
| `doc_frequency` | Number of distinct documents containing the pair |
| `pos_pattern` | Majority-vote POS pattern (e.g. `NOUN+NOUN`) |
| `pmi`, `t_score`, `logdice`, `llr` | Raw association scores |
| `*_norm` | Percentile-normalized scores |
| `combined_score` | Weighted combination (0.30/0.20/0.30/0.20) |
| `example_1` … `example_5` | Real corpus sentences containing the pair |

In [61]:
from pathlib import Path
from IPython.display import display

# Number of examples per collocation
N_EXAMPLES = CONFIG["N_EXAMPLES_PER_PAIR"]

# Add example columns
for i in range(N_EXAMPLES):
    col = f"example_{i+1}"

    best_df[col] = best_df.apply(
        lambda r, i=i: (
            pair_examples.get((r["word1"], r["word2"]), [])
            + [{}] * N_EXAMPLES
        )[i].get("sentence"),
        axis=1
    )

# Final columns
final_cols = [
    "pair_id",
    "word1",
    "word2",
    "frequency",
    "doc_frequency",
    "pos_pattern",
    "pmi",
    "t_score",
    "logdice",
    "llr",
    "pmi_norm",
    "t_score_norm",
    "logdice_norm",
    "llr_norm",
    "combined_score",
] + [f"example_{i+1}" for i in range(N_EXAMPLES)]

best_df_final = best_df[final_cols].copy()

# Output paths
final_xlsx_path = Path(CONFIG["WORKING_DIR"]) / "best_df.xlsx"
final_parquet_path = Path(CONFIG["WORKING_DIR"]) / "best_df.parquet"

# Save
best_df_final.to_excel(final_xlsx_path, index=False)
best_df_final.to_parquet(final_parquet_path, index=False)

# Information
print("=" * 80)
print("FINAL DATASET")
print("=" * 80)
print(f"Rows:    {len(best_df_final):,}")
print(f"Columns: {len(best_df_final.columns):,}")
print(f"Examples per pair: {N_EXAMPLES}")
print()
print(f"Excel:    {final_xlsx_path}")
print(f"Parquet:  {final_parquet_path}")
print("=" * 80)

# Display sample
display(
    best_df_final.head(20)
)

FINAL DATASET
Rows:    5,000
Columns: 20
Examples per pair: 5

Excel:    /kaggle/working/best_df.xlsx
Parquet:  /kaggle/working/best_df.parquet


,pair_id,word1,word2,frequency,doc_frequency,pos_pattern,pmi,t_score,logdice,llr,pmi_norm,t_score_norm,logdice_norm,llr_norm,combined_score,example_1,example_2,example_3,example_4,example_5
0,1,گفت,وگو,814,608,NOUN+NOUN,11.290361,28.519294,13.752259,13414.676885,0.982306,0.998920,0.999834,0.999751,0.994376,گل همچنین با محمدرضا عارف معاون اول رئیس جمهور...,بلافاصله جهت گفت وگو با خبرنگاران عازم سفارت ت...,در عین حال محسن امین زاده معاون امور کشورهای آ...,به بهانه این رویداد مهم اقتصادی ـ سینمایی با م...,او در گفت وگویی با روزنامه لیبراسیون گفته بود ...
1,2,دانش,آموزان,476,152,NOUN+NOUN,11.350572,21.809069,13.296765,7684.302368,0.983054,0.997176,0.998754,0.999169,0.993811,* مکلف کردن دانش آموزان برای کمک به کتابداران ...,به همین دلیل، دانش آموزان مدارس استیجاری در مد...,* نظر: خوب شد که لااقل یکی ازمسئولان آموزشی به...,مدرسه و خانه دو محیط و فضایی هستند که دانش آمو...,در شرایطی که 30 درصد مدارس تهران غیرانتفاعی اس...
2,3,جست,وجو,167,109,NOUN+NOUN,13.314680,12.921580,13.595802,3187.237164,0.995348,0.980894,0.999585,0.995099,0.993678,تکریت شاید همان نقطه قرمزی باشد که همگان تاکنو...,می‌توان علت را در میزان نیروی محرکه(اسب بخار)د...,از طرف دیگر بیگل 2 می‌تواند از طریق استنشاق هو...,طیف سنج جرمی موجود در بیگل که می‌تواند ترکیبات...,پلیس به منظور جست وجوی توزیع کنندگان مواد مخدر...
3,4,نرم,افزار,184,71,NOUN+ADJ,12.675292,13.562586,13.361171,3202.243273,0.993271,0.984217,0.998920,0.995182,0.993537,microsoft frontpage 11 بسیاری از طراحان حرفه ا...,یکی از قابلیت های جدید این نرم افزار امکان مشا...,برای آزمایش کردن صفحات طراحی شده در این نرم اف...,همچنین ابزاری برای عیبیابی کدهایی که توسط کار...,microsoft publisher 11 چون معمولا کاربران به ص...
4,5,علاقه,مندان,198,147,NOUN+NOUN,11.872055,14.067493,12.774488,3258.460507,0.988952,0.985795,0.997176,0.995348,0.992067,دوستی این دو مترجم ثمرات خوبی برای علاقه مندا...,"استقبال علاقه مندان بیضایی از"" سگ کشی ""برای س...",تقریبا همه علاقه مندان سینمای ایران در جریان م...,براساس این گزارش، پیش بینی می‌شود در این نمایش...,در حالی که بسیاری از علاقه مندان به دنبال کتاب...
5,6,وسیله,نقلیه,240,100,NOUN+ADJ,11.371549,15.486086,12.658963,3715.894425,0.983635,0.990032,0.996179,0.996096,0.991170,حتی گاهی رانندگان وسائل نقلیه آنچنان پولهای پا...,کاربردهای its را می‌توان به دو گروه اصلی «زیرس...,ژاپن استفاده از تجهیزات داخل وسیله نقلیه(12)بر...,به علاوه این تکنولوژی، اطلاعات وسایل نقلیه مسا...,نتایج حاصل از این مطالعه امکان ردیابی تمام وسا...
6,7,مرگ,میر,200,117,NOUN+NOUN,11.562464,14.137460,12.518443,3216.119923,0.986293,0.986293,0.995597,0.995265,0.990879,*واقعیت: افزایش مالیات باعث کاهش تعداد سیگاری ...,بنا به گفته وی ، حوادث ترافیکی ، علت اول یا دو...,دست کم دانشمندان امیدوارند با بهره گرفتن از چن...,دکتر افراسیاب دهلوی مصرف کنندگان دخانیات با مر...,بنابراین 5/1 میلیون قطعه مرغ آلوده به ویروس مع...
7,8,صرفه,جوی,125,81,NOUN+NOUN,13.363167,11.179279,13.446148,2300.275962,0.995846,0.968350,0.999252,0.992274,0.990655,این امر باعث صرفه جویی مقدار قابل توجهی پول و ...,وی گفت: این میزان تولید حدود 200 تا 250 میلیون...,می‌توان با آموزش همگانی به عنوان یکی از راهکار...,رعایت این امر برای جلوگیری از ایجاد بوی نامطبو...,در این حالت صرفه جویی قابل توجهی در مصرف برق ب...
8,9,بهره,بردار,508,269,NOUN+NOUN,10.581711,22.524148,12.996454,7176.992625,0.973002,0.997342,0.998006,0.998920,0.990555,این چه قضاوت اشتباهی است که نسبت به بهره بردار...,وی از بهره برداری 16 کلینیک مثلثی در 16 استان ...,یک بار در بهره برداری از کانال ژیان و این بار ...,پیش بینی می‌شود که امر بهره برداری از دو حوزه ...,هم اینک یک کنسرسیوم ژاپنی دیگر در تلاش است تا ...
9,10,آیین,نامه,285,96,NOUN+NOUN,10.922648,16.873246,12.661963,4117.088426,0.978651,0.992773,0.996262,0.997093,0.990447,تدوین آیین نامه اجرایی پرداخت تسهیلات به مالکی...,این آیین نامه که نحوه اعطای تسهیلات به مالکین ...,در این آیین نامه میزان اعتبار پرداختی به مالکی...,براساس این گزارش، آیین نامه مذکور تا پایان خرد...,به همین منظور آیین نامه ای از سوی وزارت تعاون ...


## 15. Lookup and Exploration

The lookup helper lets a user query any Persian word and retrieve its strongest collocations. It applies
the **same lemmatization logic** as the extraction pipeline, so an inflected query form still finds the
canonical entries, and results can be ranked by any single measure or by the combined score.

In [66]:
def lemmatize_word(word):
    doc = nlp(word)
    for sent in doc.sentences:
        for token in sent.words:
            return normalize_verb_lemma(
                token.lemma or token.text,
                pos=token.upos,
                surface=token.text
            )
    return word


def show_collocations(word, top_k=10, method="logdice", df=None):
    df = df if df is not None else best_df_final
    col = {
        "pmi": "pmi",
        "t_score": "t_score",
        "logdice": "logdice",
        "llr": "llr",
        "combined": "combined_score"
    }.get(method, "logdice")

    lemma = lemmatize_word(word)

    matches = df[
        (df["word1"] == lemma) |
        (df["word2"] == lemma)
    ].sort_values(col, ascending=False).head(top_k)

    if matches.empty:
        print(f"No collocations found for '{word}' (lemma='{lemma}').")
        return matches

    print(f"Top {len(matches)} collocations for '{word}' ranked by {method}:")
    return matches[
        ["word1","word2","pos_pattern","frequency",
         "doc_frequency","pmi","t_score","logdice","llr","combined_score",
         "example_1"]
    ]

## 16. Ranking Analysis

The 5,000 final collocations are grouped into consecutive bands of 1,000 by combined score. Per-band
max/min/mean confidence summarizes how discriminative the ranking is, and random samples from each band
allow qualitative inspection of quality degradation toward the tail.

In [69]:
# Sort from highest to lowest confidence
df = best_df_final.sort_values("combined_score", ascending=False).copy()

# Create groups of 1000 collocations
df["score_group"] = np.arange(len(df)) // 1000 + 1

# Summary of each 1000-item confidence range
summary = (
    df.groupby("score_group")
    .agg(
        Collocations=("combined_score", "size"),
        Max_Confidence=("combined_score", "max"),
        Min_Confidence=("combined_score", "min"),
        Mean_Confidence=("combined_score", "mean"),
    )
)

summary.index = [f"Top {i*1000+1:,}–{min((i+1)*1000, len(df)):,}" 
                 for i in range(len(summary))]

display(
    summary.style.format({
        "Max_Confidence": "{:.4f}",
        "Min_Confidence": "{:.4f}",
        "Mean_Confidence": "{:.4f}",
    })
)

# Show a small random sample from each 1000-item group
cols = ["word1", "word2", "frequency", "combined_score", "example_1"]

for group_id, group in df.groupby("score_group"):
    start = (group_id - 1) * 1000 + 1
    end = min(group_id * 1000, len(df))

    print(f"\n{'='*90}")
    print(
        f"CONFIDENCE GROUP {group_id} | "
        f"Ranks {start:,}–{end:,} | "
        f"Confidence: {group['combined_score'].min():.4f} → "
        f"{group['combined_score'].max():.4f}"
    )
    print(f"{'='*90}")

    display(
        group.sample(
            min(10, len(group)),
            random_state=CONFIG["SEED"]
        )[cols].sort_values("combined_score", ascending=False)
    )

,Collocations,Max_Confidence,Min_Confidence,Mean_Confidence
"Top 1–1,000",1000,0.9944,0.8750,0.9230
"Top 1,001–2,000",1000,0.8746,0.7958,0.8334
"Top 2,001–3,000",1000,0.7957,0.7250,0.7603
"Top 3,001–4,000",1000,0.7249,0.6540,0.6896
"Top 4,001–5,000",1000,0.6539,0.5850,0.6193



CONFIDENCE GROUP 1 | Ranks 1–1,000 | Confidence: 0.8750 → 0.9944


,word1,word2,frequency,combined_score,example_1
136,سیگار,کشیدن,151,0.962776,سیگار کشیدن: اعتیاد به سیگار بر خطر بیماری های...
411,هرچه,سریع,39,0.929465,با گسترش احتمال خرابکاری، اکنون آمریکا بیش از ...
514,سانحه,رانندگی,34,0.917420,در سال گذشته بیش از چهار هزار نفر در جریان سوا...
521,روز,جمعه,138,0.916697,گروه ورزشی: هفته پایانی از مرحله مقدماتی مسابق...
626,غنی,اورانیوم,25,0.906687,این دستگاه ها در نهایت در یک کارخانه عظیم غنی ...
660,دریافت,وام,61,0.904353,به نظر او ممکن است بسیاری از زوج ها ازشهرهای ن...
678,راه,آهن,78,0.903182,در حال حاضر در این کشورها راه و راه آهن بسیار ...
737,ادامه,تحصیل,78,0.895473,در واقع در یک چنین وضعیتی افراد دچار نوعی جبر ...
740,پدید,آوردن,46,0.894941,هر جریان انواع مشخصی از خیر خاص خود را پدید آو...
859,نادیده,گرفتن,106,0.886435,ولی واقعا چرا باید تهیه کنندگان درصدی از فروش ...



CONFIDENCE GROUP 2 | Ranks 1,001–2,000 | Confidence: 0.7958 → 0.8746


,word1,word2,frequency,combined_score,example_1
1136,تصویب,قانون,67,0.861979,افزایش قابل توجه حجم تقاضا برای سرمایه گذاری خ...
1411,عامل,مهم,82,0.838769,این فجایع نسبت به تخریب زمین - به عنوان یکی از...
1513,انتشار,بیانیه,22,0.831052,به این ترتیب در دوم اردیبهشت 1358 سپاه پاسدارا...
1521,برگزاری,کنگره,27,0.830645,دبیر کنگره بوعلی سینا می‌گوید: «برگزاری این کن...
1626,فیلم,سینما,164,0.823027,با تمام این اوصاف پرمشکل ترین فیلم های سینمای ...
1660,بازی,تیم,91,0.821208,بازی این دو تیم دیدن دارد، اما احتمالا برای اص...
1678,پنجه,نرم,16,0.820169,کشور گرم و خشک پاکستان نیاز شدیدی به آب دارد، ...
1737,دستور,سهم,53,0.815235,این گزارش حاکی است: در چنین شرایطی بیشترین تعد...
1740,بهره,رسیدن,94,0.815052,براین اساس، عملیات اجرایی این طرح شروع و پیش ب...
1859,اعلام,آمادگی,42,0.806720,و بیمارستان میلاد نیز به ستاد بحران وزارت بهدا...


...

,word1,word2,frequency,combined_score,example_1
2136,ذخیره,ساز,32,0.786443,3- کادر بعدی شامل محل ذخیره سازی فایل تصویری و...
2411,گزارش,تلویزیونی,37,0.765792,به گزارش شبکه تلویزیونی آرته، آثار هنری دهه ها...
2513,پروانه,ساخت,20,0.758789,ایرنا: شورای صدور پروانه فیلمسازی 35 میلیمتری ...
2521,وزیر,کشور,149,0.758207,• ضرورت تشکیل جلسات مشترک رهبران مذهبی فرانسه ...
2626,پیشگیری,درمان,26,0.750831,بنابراین، استفاده از ورزش به منظور پیشگیری و د...
2660,درصد,صادرات,59,0.748231,بخش صنعت با 8/1579 میلیون دلار صادرات در این س...
2678,وهله,اول,19,0.747279,در وهله اول، هر شاخص قاعده ای برای اندازه گیری...
2737,دچار,تغییر,30,0.743620,به تازگی رئیس فدراسیون شمشیربازی نیز دچار چنین...
2740,اثر,انفجار,29,0.743446,در کشور خودمان هم هرازگاهی افرادی بی گناه بر ا...
2859,تحصیل,عالی,23,0.735305,این همه دختر که وارد دانشگاه می‌شوند و با سطح ...



CONFIDENCE GROUP 4 | Ranks 3,001–4,000 | Confidence: 0.6540 → 0.7249


,word1,word2,frequency,combined_score,example_1
3136,مدیر,ستاد,25,0.714595,عباسی مدیر ستاد برگزاری مسابقه می‌گوید: هدف از...
3411,نگاه,کردن,255,0.695755,او با فیلم های گاهی به آسمان نگاه کن و ستاره ...
3513,آینده,خبر,37,0.689276,همچنین اخیرا نایب رئیس کانون سردفتران و دفتریا...
3521,مدیر,مسئول,31,0.688121,کارکنان روزنامه الوطن اظهار داشتند که حکم اخرا...
3626,اصلاح,اقتصادی,28,0.681143,اولویت او به جای بقا، انجام اصلاحات اقتصادی است.
3660,تامین,انرژی,24,0.678256,این نیرو عامل اصلی تامین کننده انرژی لازم برای...
3678,کیفیت,بالا,23,0.676367,clone cd کیفیت بالایی را در کپی کردن سی دی ارا...
3737,تهیه,فیلم,50,0.672437,اما مسئله از آن جا آغاز می‌شود که از درآمد حاص...
3740,ساخت,مسکونی,17,0.672259,محمد سعیدی کیا افزود: تاکنون 10 تشکل غیردولتی ...
3859,علم,اجتماعی,32,0.664055,8- مقدمه ای بر جامعه شناسی خانواده،ص 61 9- دای...



CONFIDENCE GROUP 5 | Ranks 4,001–5,000 | Confidence: 0.5850 → 0.6539


,word1,word2,frequency,combined_score,example_1
4136,تهران,آمدن,85,0.643886,البته صمد زارع که شب قبل از تمرین با هواپیما ب...
4411,بازار,مسکن,23,0.625486,از سوی دیگر نمایندگان مجلس هم حدود 3 ماه گذشته...
4513,بیرون,رفتن,23,0.618932,شوت نویدکیا در دقیقه 65 یکی از آنها بود که با ...
4521,سلامت,انسان,18,0.618483,این تحقیق نشان می‌دهد که خواب کافی یک مسئله تف...
4626,بار,آوردن,31,0.610641,خرازی نیز در این دیدار در مورد تحولات منطقه گف...
4660,بودجه,مجلس,24,0.608606,گسترش مترو در بودجه سال 82 مجلس به دولت اجازه ...
4678,ریز,انجام,30,0.607651,از مدتها قبل در این خصوص اقدام و برنامه ریزی ه...
4738,مرز,ریال,17,0.603422,ارزش بازار سهام سیمان فارس و خوزستان در پایان ...
4740,کشور,اسلامی,87,0.603306,وی اضافه کرد: کشورهای اسلامی تنها به صدور بیان...
4859,نخستین,روز,32,0.594899,از نخستین روزهای جدی شدن احتمال حمله آمریکا به...


## 17. Checkpoints and Reproducibility

All intermediate outputs are stored in `/kaggle/working/checkpoints_stanza/` so the pipeline can resume
after interruption. Final deliverables are written to `/kaggle/working/`.

Key checkpoints:
- `corpus_raw.parquet`
- `corpus_normalized.parquet`
- `raw_sentence_pool.jsonl`
- `inspection_df.csv`
- `sentences_raw.jsonl` — main full-corpus checkpoint (tagged sentences + per-document `doc_done` markers)
- `candidates_raw.csv`
- `candidates_filtered.csv`
- `best_df_scored.parquet`
- `pair_examples.json`
- `examples_progress.json`
- `best_df.xlsx`
- `pair_examples_long.csv`

Reproducibility: all randomness is seeded (`SEED = 42`), the 50-document evaluation is deterministic,
and every stage persists its output so the notebook can be resumed stage-by-stage after interruption.
The default configuration processes the complete corpus after the 50-document quality checks pass.

## Final Ranking Validation with Min-Max Scaling

This is a **separate validation/re-ranking step** applied to the already generated 5,000 final candidates
in `best_df.xlsx`. It does not modify the percentile-based ranking above. Each association metric is
Min-Max scaled independently, combined with the same weights (PMI 0.30, t-score 0.20, logDice 0.30,
LLR 0.20), sorted, and saved as `best_df_minmax.xlsx` / `best_df_minmax.parquet`. The original
`best_df.xlsx` is left untouched.

In [3]:
# Final ranking validation: Min-Max scaling over the existing 5,000 candidates
from pathlib import Path
import numpy as np
import pandas as pd

#minmax_df = pd.read_excel(Path(CONFIG["WORKING_DIR"]) / "best_df.xlsx")
minmax_df = pd.read_excel("best_df.xlsx")

weights = {"pmi": 0.30, "t_score": 0.20, "logdice": 0.30, "llr": 0.20}
assert abs(sum(weights.values()) - 1.0) < 1e-9

for m, w in weights.items():
    col = minmax_df[m].astype(float)
    rng = col.max() - col.min()
    minmax_df[f"{m}_minmax"] = (col - col.min()) / rng if rng > 0 else 0.0

minmax_df["minmax_score"] = sum(
    minmax_df[f"{m}_minmax"] * w for m, w in weights.items()
)

minmax_df = minmax_df.sort_values("minmax_score", ascending=False).reset_index(drop=True)
minmax_df["pair_id"] = np.arange(1, len(minmax_df) + 1)

display_cols = [
    "pair_id", "word1", "word2", "frequency", "doc_frequency", "pos_pattern",
    "pmi", "t_score", "logdice", "llr", "minmax_score",
]
display(minmax_df[display_cols].head(20))

minmax_xlsx_path = Path("best_df_minmax.xlsx")
minmax_parquet_path = Path("best_df_minmax.parquet")
minmax_df.to_excel(minmax_xlsx_path, index=False)
minmax_df.to_parquet(minmax_parquet_path, index=False)

print(f"Rows: {len(minmax_df):,}")
print(f"Excel:   {minmax_xlsx_path}")
print(f"Parquet: {minmax_parquet_path}")

,pair_id,word1,word2,frequency,doc_frequency,pos_pattern,pmi,t_score,logdice,llr,minmax_score
0,1,قرار,گرفتن,2419,1483,NOUN+VERB,7.129518,48.832078,12.434250,21065.724722,0.749511
1,2,گفت,وگو,814,608,NOUN+NOUN,11.290361,28.519294,13.752259,13414.676885,0.714176
2,3,سرمایه,گذاری,996,413,NOUN+ADJ,9.737569,31.522499,13.115688,12867.242444,0.669331
3,4,نشان,دادن,1787,1053,NOUN+VERB,7.417329,42.025623,11.842417,17344.689698,0.667917
4,5,وجود,داشتن,2388,1263,NOUN+VERB,6.136455,48.172526,11.531316,17486.522435,0.660630
5,6,دانش,آموزان,476,152,NOUN+NOUN,11.350572,21.809069,13.296765,7684.302368,0.614383
6,7,هرج,مرج,17,17,NOUN+NOUN,17.261053,4.123079,13.958180,435.010455,0.604543
7,8,حال,حاضر,1068,727,NOUN+ADJ,8.391849,32.582975,12.294788,11401.078177,0.604416
8,9,تمامیت,ارضی,22,17,NOUN+ADJ,16.546113,4.690367,13.787006,520.935081,0.587906
9,10,بهره,بردار,508,269,NOUN+NOUN,10.581711,22.524148,12.996454,7176.992625,0.587033


Rows: 5,000
Excel:   best_df_minmax.xlsx
Parquet: best_df_minmax.parquet


## **Conclusion**

This notebook presented a complete, reproducible pipeline for extracting statistically strong Persian bigram collocations from the Hamshahri corpus. Raw article text is preserved throughout, Persian-specific normalization and verb lemma canonicalization are applied, PERSON entities are filtered out via character-offset-aligned NER, and candidates are ranked by a weighted combination of PMI, t-score, logDice, and LLR — with a separate Min-Max validation ranking at the end.

**Deliverables:**
- `best_df.xlsx` / `best_df.parquet` — 5,000 ranked collocations with scores, POS patterns, and up to 5 real corpus example sentences each
- `best_df_minmax.xlsx` / `best_df_minmax.parquet` — Min-Max-scaled validation ranking
- All intermediate checkpoints under `checkpoints_stanza/` for full reproducibility

The deterministic seed, staged quality evaluation, and resumable checkpoints make every result in this notebook reproducible from the raw `.ham` corpus files.